# ALMA Archive Data-Relationship Validation

## Objective

This notebook tests whether the data relationships observed in the
Centaurus A case study generalize to other ALMA Archive datasets.

The unit of sampling is the Member OUS rather than an individual Archive
row. Complete science-observation rows will be retrieved for every selected
Member OUS before its internal structure is evaluated.

This notebook investigates Archive data semantics and granularity. It does
not implement formal duplication rules.

## Hypotheses from Notebook 1

The following sample-derived hypotheses will be tested:

1. `member_ous_uid` is a suitable grouping key for an independent dataset.
2. Dataset-level metadata are internally consistent within a Member OUS.
3. Archive rows are consistent with source–spectral-window-related records.
4. Archive-row count, parsed SPW count, and `frequency_support` interval count
   are usually equal.
5. A Member OUS usually contains one source, but this must not be assumed.
6. A Member OUS may contain a variable number of spectral windows.
7. Archive `bandwidth` and `frequency_support` width are related but are not
   necessarily interchangeable.
8. Polygon footprints are not equivalent to mosaic observations.
9. Member OUS, Group OUS, and ASDM identifiers must remain separate.

The hypotheses will be classified as:

- Supported in the current sample
- Conditional
- Rejected
- Not yet tested

In [1]:
import re

import numpy as np
import pandas as pd
import pyvo

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

TAP_URL = "https://almascience.eso.org/tap"

service = pyvo.dal.TAPService(TAP_URL)

print("Connected to:", TAP_URL)

Connected to: https://almascience.eso.org/tap


In [2]:
def run_tap_query(query, maxrec=10_000):
    """Run an ADQL query and return a Pandas DataFrame."""

    response = service.search(
        query,
        maxrec=maxrec,
    )

    return response.to_table().to_pandas()

In [3]:
sample_strata = [
    {
        "label": "band3_non_mosaic",
        "frequency_min_ghz": 84,
        "frequency_max_ghz": 116,
        "is_mosaic": "F",
    },
    {
        "label": "band3_mosaic",
        "frequency_min_ghz": 84,
        "frequency_max_ghz": 116,
        "is_mosaic": "T",
    },
    {
        "label": "band6_non_mosaic",
        "frequency_min_ghz": 211,
        "frequency_max_ghz": 275,
        "is_mosaic": "F",
    },
    {
        "label": "band6_mosaic",
        "frequency_min_ghz": 211,
        "frequency_max_ghz": 275,
        "is_mosaic": "T",
    },
    {
        "label": "band7_non_mosaic",
        "frequency_min_ghz": 275,
        "frequency_max_ghz": 373,
        "is_mosaic": "F",
    },
    {
        "label": "band7_mosaic",
        "frequency_min_ghz": 275,
        "frequency_max_ghz": 373,
        "is_mosaic": "T",
    },
    {
        "label": "band9_non_mosaic",
        "frequency_min_ghz": 602,
        "frequency_max_ghz": 720,
        "is_mosaic": "F",
    },
    {
        "label": "band9_mosaic",
        "frequency_min_ghz": 602,
        "frequency_max_ghz": 720,
        "is_mosaic": "T",
    },
]

In [4]:
candidate_tables = []

for stratum in sample_strata:
    candidate_query = f"""
    SELECT DISTINCT TOP 12
        member_ous_uid,
        proposal_id,
        is_mosaic
    FROM ivoa.obscore
    WHERE science_observation = 'T'
    AND member_ous_uid IS NOT NULL
    AND frequency >= {stratum["frequency_min_ghz"]}
    AND frequency < {stratum["frequency_max_ghz"]}
    AND is_mosaic = '{stratum["is_mosaic"]}'
    """

    stratum_candidates = run_tap_query(
        candidate_query,
        maxrec=100,
    )

    stratum_candidates["sample_stratum"] = (
        stratum["label"]
    )

    candidate_tables.append(stratum_candidates)

candidate_member_df = (
    pd.concat(
        candidate_tables,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "sample_stratum",
            "member_ous_uid",
        ]
    )
)

display(candidate_member_df)

,member_ous_uid,proposal_id,is_mosaic,sample_stratum
0,uid://A001/X12d/X6,2013.1.00266.S,F,band3_non_mosaic
1,uid://A001/X12d/Xa,2013.1.00266.S,F,band3_non_mosaic
2,uid://A001/X132/Xa,2013.1.00833.S,F,band3_non_mosaic
3,uid://A001/X136/X6,2013.1.01383.S,F,band3_non_mosaic
4,uid://A001/X136/X8,2013.1.01383.S,F,band3_non_mosaic
...,...,...,...,...
91,uid://A001/X74/X172,2011.0.00084.S,T,band9_mosaic
92,uid://A001/X122/X407,2013.1.00126.S,T,band9_mosaic
93,uid://A001/X122/X409,2013.1.00126.S,T,band9_mosaic
94,uid://A001/X122/X411,2013.1.00126.S,T,band9_mosaic


In [5]:
candidate_counts = (
    candidate_member_df
    .groupby("sample_stratum")
    .size()
    .rename("candidate_member_count")
    .to_frame()
)

display(candidate_counts)

,candidate_member_count
sample_stratum,
band3_mosaic,12
band3_non_mosaic,12
band6_mosaic,12
band6_non_mosaic,12
band7_mosaic,12
band7_non_mosaic,12
band9_mosaic,12
band9_non_mosaic,12


In [6]:
selected_groups = []

for stratum, group in candidate_member_df.groupby(
    "sample_stratum"
):
    unique_group = (
        group
        .drop_duplicates("member_ous_uid")
        .reset_index(drop=True)
    )

    selected_count = min(3, len(unique_group))

    selected_groups.append(
        unique_group.sample(
            n=selected_count,
            random_state=42,
        )
    )

selected_member_df = (
    pd.concat(
        selected_groups,
        ignore_index=True,
    )
    .sort_values(
        [
            "sample_stratum",
            "proposal_id",
        ]
    )
    .reset_index(drop=True)
)

display(selected_member_df)

print(
    "Selected Member OUS datasets:",
    selected_member_df["member_ous_uid"].nunique(),
)

,member_ous_uid,proposal_id,is_mosaic,sample_stratum
0,uid://A001/X6f/X4,2011.0.00474.S,T,band3_mosaic
1,uid://A001/X11f/X98,2013.1.00041.S,T,band3_mosaic
2,uid://A001/X8aa/Xe,2016.1.00801.S,T,band3_mosaic
3,uid://A001/X12d/X6,2013.1.00266.S,F,band3_non_mosaic
4,uid://A001/X2d8/X5,2015.1.00030.S,F,band3_non_mosaic
5,uid://A001/X340/X6,2015.1.00040.S,F,band3_non_mosaic
6,uid://A001/X6f/Xe,2011.0.00131.S,T,band6_mosaic
7,uid://A001/X892/Xa,2016.1.01013.S,T,band6_mosaic
8,uid://A001/X892/X8,2016.1.01013.S,T,band6_mosaic
9,uid://A001/X62/Xb,2011.0.00780.S,F,band6_non_mosaic


Selected Member OUS datasets: 24


In [7]:
archive_columns = [
    "proposal_id",
    "group_ous_uid",
    "member_ous_uid",
    "obs_id",
    "asdm_uid",
    "target_name",
    "s_ra",
    "s_dec",
    "s_region",
    "frequency",
    "bandwidth",
    "frequency_support",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "antenna_arrays",
    "is_mosaic",
    "band_list",
    "obs_release_date",
]

In [8]:
selected_member_uids = (
    selected_member_df["member_ous_uid"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

member_uid_sql = ",\n        ".join(
    f"'{member_uid}'"
    for member_uid in selected_member_uids
)

print("Number of selected UIDs:", len(selected_member_uids))

Number of selected UIDs: 24


In [9]:
complete_member_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
        {member_uid_sql}
)
"""

count_result_df = run_tap_query(
    complete_member_count_query,
    maxrec=1,
)

expected_row_count = int(
    count_result_df["total_rows"].iloc[0]
)

print("Expected Archive rows:", expected_row_count)

Expected Archive rows: 193


In [10]:
selected_columns_sql = ",\n    ".join(
    archive_columns
)

complete_member_query = f"""
SELECT
    {selected_columns_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
        {member_uid_sql}
)
"""

complete_member_df = run_tap_query(
    complete_member_query,
    maxrec=max(expected_row_count, 1),
)

print("Expected rows:", expected_row_count)
print("Retrieved rows:", len(complete_member_df))
print(
    "Complete retrieval:",
    len(complete_member_df) == expected_row_count,
)

Expected rows: 193
Retrieved rows: 193
Complete retrieval: True


In [11]:
candidate_dataset_fields = [
    "proposal_id",
    "group_ous_uid",
    "asdm_uid",
    "target_name",
    "s_ra",
    "s_dec",
    "s_region",
    "spatial_resolution",
    "antenna_arrays",
    "is_mosaic",
    "cont_sensitivity_bandwidth",
    "frequency_support",
    "band_list",
    "obs_release_date",
]

In [12]:
member_field_cardinality = (
    complete_member_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )[candidate_dataset_fields]
    .nunique(dropna=False)
)

In [13]:
field_conflict_summary = (
    member_field_cardinality
    .gt(1)
    .sum()
    .sort_values(ascending=False)
    .rename("member_ous_with_multiple_values")
    .to_frame()
)

display(field_conflict_summary)

,member_ous_with_multiple_values
target_name,4
spatial_resolution,4
cont_sensitivity_bandwidth,4
s_ra,3
s_dec,3
s_region,3
frequency_support,2
asdm_uid,1
antenna_arrays,1
proposal_id,0


In [14]:
conflicting_members = (
    member_field_cardinality
    .gt(1)
    .any(axis=1)
)

display(
    member_field_cardinality.loc[
        conflicting_members
    ]
)

,proposal_id,group_ous_uid,asdm_uid,target_name,s_ra,s_dec,s_region,spatial_resolution,antenna_arrays,is_mosaic,cont_sensitivity_bandwidth,frequency_support,band_list,obs_release_date
member_ous_uid,,,,,,,,,,,,,,
uid://A001/X13e/Xc,1,1,1,4,4,4,4,4,1,1,4,1,1,1
uid://A001/X21f/Xb,1,1,1,15,15,14,15,15,1,1,15,1,1,1
uid://A001/X5a4/Xe,1,1,1,4,4,4,4,4,1,1,4,3,1,1
uid://A001/X62/X6,1,1,2,2,1,1,1,2,2,1,2,2,1,1


In [15]:
member_identifier_summary = (
    complete_member_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )
    .agg(
        archive_rows=("obs_id", "size"),
        proposal_count=("proposal_id", "nunique"),
        group_ous_count=(
            "group_ous_uid",
            "nunique",
        ),
        asdm_count=("asdm_uid", "nunique"),
        target_name_count=(
            "target_name",
            "nunique",
        ),
        mosaic_state_count=(
            "is_mosaic",
            "nunique",
        ),
        frequency_support_count=(
            "frequency_support",
            "nunique",
        ),
    )
    .reset_index()
)

display(member_identifier_summary)

,member_ous_uid,archive_rows,proposal_count,group_ous_count,asdm_count,target_name_count,mosaic_state_count,frequency_support_count
0,uid://A001/X11e/Xb,4,1,1,1,1,1,1
1,uid://A001/X11f/X98,4,1,1,1,1,1,1
2,uid://A001/X120/X17,4,1,1,1,1,1,1
3,uid://A001/X122/X409,4,1,1,1,1,1,1
4,uid://A001/X122/X411,4,1,1,1,1,1,1
5,uid://A001/X125/X5,4,1,1,1,1,1,1
6,uid://A001/X12d/X6,11,1,1,1,1,1,1
7,uid://A001/X13e/Xb8,4,1,1,1,1,1,1
8,uid://A001/X13e/Xba,4,1,1,1,1,1,1
9,uid://A001/X13e/Xc,16,1,1,1,4,1,1


In [16]:
display(
    member_identifier_summary.loc[
        (
            member_identifier_summary[
                "group_ous_count"
            ] > 1
        )
        |
        (
            member_identifier_summary[
                "asdm_count"
            ] > 1
        )
        |
        (
            member_identifier_summary[
                "target_name_count"
            ] > 1
        )
        |
        (
            member_identifier_summary[
                "frequency_support_count"
            ] > 1
        )
    ]
)

,member_ous_uid,archive_rows,proposal_count,group_ous_count,asdm_count,target_name_count,mosaic_state_count,frequency_support_count
9,uid://A001/X13e/Xc,16,1,1,1,4,1,1
13,uid://A001/X21f/Xb,45,1,1,1,15,1,1
16,uid://A001/X5a4/Xe,16,1,1,1,4,1,3
17,uid://A001/X62/X6,8,1,1,2,2,1,2


In [17]:
obs_id_pattern = re.compile(
    r"\.source\."
    r"(?P<obs_id_source>.+)"
    r"\.spw\."
    r"(?P<spw_identifier>[^.]+)$"
)

parsed_obs_id_df = (
    complete_member_df["obs_id"]
    .astype("string")
    .str.extract(obs_id_pattern)
)

relationship_analysis_df = pd.concat(
    [
        complete_member_df.reset_index(drop=True),
        parsed_obs_id_df.reset_index(drop=True),
    ],
    axis=1,
)

In [18]:
unparsed_obs_id_df = (
    relationship_analysis_df.loc[
        relationship_analysis_df[
            "spw_identifier"
        ].isna()
    ]
)

print(
    "Unparsed obs_id rows:",
    len(unparsed_obs_id_df),
)

display(
    unparsed_obs_id_df[
        [
            "member_ous_uid",
            "obs_id",
            "proposal_id",
        ]
    ]
)

Unparsed obs_id rows: 0


,member_ous_uid,obs_id,proposal_id


In [19]:
frequency_interval_pattern = re.compile(
    r"\[\s*"
    r"(?P<lower>[0-9.+\-Ee]+)"
    r"\.\."
    r"(?P<upper>[0-9.+\-Ee]+)"
    r"\s*GHz"
)

def count_frequency_support_intervals(value):
    if pd.isna(value):
        return pd.NA

    return len(
        frequency_interval_pattern.findall(
            str(value)
        )
    )

relationship_analysis_df[
    "support_interval_count"
] = (
    relationship_analysis_df[
        "frequency_support"
    ]
    .apply(count_frequency_support_intervals)
)

In [20]:
member_structure_summary = (
    relationship_analysis_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )
    .agg(
        archive_row_count=("obs_id", "size"),
        unique_obs_id_count=(
            "obs_id",
            "nunique",
        ),
        parsed_source_count=(
            "obs_id_source",
            "nunique",
        ),
        spw_identifier_count=(
            "spw_identifier",
            "nunique",
        ),
        support_interval_min=(
            "support_interval_count",
            "min",
        ),
        support_interval_max=(
            "support_interval_count",
            "max",
        ),
        frequency_support_count=(
            "frequency_support",
            "nunique",
        ),
    )
    .reset_index()
)

In [21]:
member_structure_summary[
    "rows_equal_spw_count"
] = (
    member_structure_summary[
        "archive_row_count"
    ]
    == member_structure_summary[
        "spw_identifier_count"
    ]
)

member_structure_summary[
    "spw_count_equals_support_intervals"
] = (
    member_structure_summary[
        "spw_identifier_count"
    ]
    == member_structure_summary[
        "support_interval_min"
    ]
)

member_structure_summary[
    "support_interval_count_consistent"
] = (
    member_structure_summary[
        "support_interval_min"
    ]
    == member_structure_summary[
        "support_interval_max"
    ]
)

display(member_structure_summary)

,member_ous_uid,archive_row_count,unique_obs_id_count,parsed_source_count,spw_identifier_count,support_interval_min,support_interval_max,frequency_support_count,rows_equal_spw_count,spw_count_equals_support_intervals,support_interval_count_consistent
0,uid://A001/X11e/Xb,4,4,1,4,4,4,1,True,True,True
1,uid://A001/X11f/X98,4,4,1,4,4,4,1,True,True,True
2,uid://A001/X120/X17,4,4,1,4,4,4,1,True,True,True
3,uid://A001/X122/X409,4,4,1,4,4,4,1,True,True,True
4,uid://A001/X122/X411,4,4,1,4,4,4,1,True,True,True
5,uid://A001/X125/X5,4,4,1,4,4,4,1,True,True,True
6,uid://A001/X12d/X6,11,11,1,11,11,11,1,True,True,True
7,uid://A001/X13e/Xb8,4,4,1,4,4,4,1,True,True,True
8,uid://A001/X13e/Xba,4,4,1,4,4,4,1,True,True,True
9,uid://A001/X13e/Xc,16,16,4,4,4,4,1,False,True,True


In [22]:
structure_issues_df = (
    member_structure_summary.loc[
        (
            member_structure_summary[
                "parsed_source_count"
            ] > 1
        )
        |
        (
            ~member_structure_summary[
                "rows_equal_spw_count"
            ]
        )
        |
        (
            ~member_structure_summary[
                "spw_count_equals_support_intervals"
            ]
        )
        |
        (
            ~member_structure_summary[
                "support_interval_count_consistent"
            ]
        )
        |
        (
            member_structure_summary[
                "frequency_support_count"
            ] > 1
        )
    ]
)

display(structure_issues_df)

print(
    "Members requiring investigation:",
    len(structure_issues_df),
)

,member_ous_uid,archive_row_count,unique_obs_id_count,parsed_source_count,spw_identifier_count,support_interval_min,support_interval_max,frequency_support_count,rows_equal_spw_count,spw_count_equals_support_intervals,support_interval_count_consistent
9,uid://A001/X13e/Xc,16,16,4,4,4,4,1,False,True,True
13,uid://A001/X21f/Xb,45,45,15,3,3,3,1,False,True,True
16,uid://A001/X5a4/Xe,16,16,4,4,4,4,3,False,True,True
17,uid://A001/X62/X6,8,8,2,4,4,4,2,False,True,True


Members requiring investigation: 4


In [23]:
member_structure_summary[
    "expected_source_spw_rows"
] = (
    member_structure_summary[
        "parsed_source_count"
    ]
    * member_structure_summary[
        "spw_identifier_count"
    ]
)

member_structure_summary[
    "rows_equal_source_spw_product"
] = (
    member_structure_summary[
        "archive_row_count"
    ]
    == member_structure_summary[
        "expected_source_spw_rows"
    ]
)

display(
    member_structure_summary[
        [
            "member_ous_uid",
            "archive_row_count",
            "parsed_source_count",
            "spw_identifier_count",
            "expected_source_spw_rows",
            "rows_equal_source_spw_product",
        ]
    ]
)

print(
    "Members matching source × SPW structure:",
    member_structure_summary[
        "rows_equal_source_spw_product"
    ].sum(),
    "/",
    len(member_structure_summary),
)

,member_ous_uid,archive_row_count,parsed_source_count,spw_identifier_count,expected_source_spw_rows,rows_equal_source_spw_product
0,uid://A001/X11e/Xb,4,1,4,4,True
1,uid://A001/X11f/X98,4,1,4,4,True
2,uid://A001/X120/X17,4,1,4,4,True
3,uid://A001/X122/X409,4,1,4,4,True
4,uid://A001/X122/X411,4,1,4,4,True
5,uid://A001/X125/X5,4,1,4,4,True
6,uid://A001/X12d/X6,11,1,11,11,True
7,uid://A001/X13e/Xb8,4,1,4,4,True
8,uid://A001/X13e/Xba,4,1,4,4,True
9,uid://A001/X13e/Xc,16,4,4,16,True


Members matching source × SPW structure: 24 / 24


In [24]:
multi_source_member_uids = (
    member_structure_summary.loc[
        member_structure_summary[
            "parsed_source_count"
        ] > 1,
        "member_ous_uid",
    ]
)

for member_uid in multi_source_member_uids:
    member_rows = relationship_analysis_df.loc[
        relationship_analysis_df[
            "member_ous_uid"
        ] == member_uid
    ]

    print("\nMember OUS:", member_uid)

    source_spw_table = pd.crosstab(
        member_rows["obs_id_source"],
        member_rows["spw_identifier"],
    )

    display(source_spw_table)


Member OUS: uid://A001/X13e/Xc


spw_identifier,11,13,15,9
obs_id_source,,,,
ADFS_17,1,1,1,1
ADFS_27,1,1,1,1
ADFS_31,1,1,1,1
ADFS_33,1,1,1,1



Member OUS: uid://A001/X21f/Xb


spw_identifier,15,17,19
obs_id_source,,,
FMR2006_1,1,1,1
FMR2006_10,1,1,1
FMR2006_11,1,1,1
FMR2006_12,1,1,1
FMR2006_13,1,1,1
FMR2006_14,1,1,1
FMR2006_15,1,1,1
FMR2006_2,1,1,1
FMR2006_3,1,1,1



Member OUS: uid://A001/X5a4/Xe


spw_identifier,16,18,20,22
obs_id_source,,,,
15470-5419c1,1,1,1,1
15470-5419c3,1,1,1,1
15557-5215c2,1,1,1,1
15557-5215c3,1,1,1,1



Member OUS: uid://A001/X62/X6


spw_identifier,11,13,15,9
obs_id_source,,,,
HD 21997,1,1,1,1
HD21997,1,1,1,1


In [25]:
def frequency_range_signature(value):
    if pd.isna(value):
        return None

    matches = frequency_interval_pattern.findall(
        str(value)
    )

    return tuple(
        (
            float(lower),
            float(upper),
        )
        for lower, upper in matches
    )

relationship_analysis_df[
    "frequency_range_signature"
] = (
    relationship_analysis_df[
        "frequency_support"
    ]
    .apply(frequency_range_signature)
)

frequency_representation_summary = (
    relationship_analysis_df
    .groupby("member_ous_uid")
    .agg(
        full_support_string_count=(
            "frequency_support",
            "nunique",
        ),
        frequency_range_signature_count=(
            "frequency_range_signature",
            "nunique",
        ),
    )
    .reset_index()
)

display(
    frequency_representation_summary.loc[
        (
            frequency_representation_summary[
                "full_support_string_count"
            ] > 1
        )
        |
        (
            frequency_representation_summary[
                "frequency_range_signature_count"
            ] > 1
        )
    ]
)

,member_ous_uid,full_support_string_count,frequency_range_signature_count
16,uid://A001/X5a4/Xe,3,2
17,uid://A001/X62/X6,2,2


In [26]:
def unique_values_as_tuple(series):
    return tuple(
        pd.unique(
            series.dropna()
        ).tolist()
    )

source_context_summary = (
    relationship_analysis_df
    .groupby(
        [
            "member_ous_uid",
            "obs_id_source",
        ],
        dropna=False,
    )
    .agg(
        archive_row_count=("obs_id", "size"),
        spw_count=(
            "spw_identifier",
            "nunique",
        ),
        target_names=(
            "target_name",
            unique_values_as_tuple,
        ),
        asdm_uids=(
            "asdm_uid",
            unique_values_as_tuple,
        ),
        ra_values=(
            "s_ra",
            unique_values_as_tuple,
        ),
        dec_values=(
            "s_dec",
            unique_values_as_tuple,
        ),
        region_count=(
            "s_region",
            "nunique",
        ),
        spatial_resolutions=(
            "spatial_resolution",
            unique_values_as_tuple,
        ),
        antenna_arrays_values=(
            "antenna_arrays",
            unique_values_as_tuple,
        ),
        support_string_count=(
            "frequency_support",
            "nunique",
        ),
        frequency_range_signature_count=(
            "frequency_range_signature",
            "nunique",
        ),
    )
    .reset_index()
)

display(
    source_context_summary.loc[
        source_context_summary[
            "member_ous_uid"
        ].isin(multi_source_member_uids)
    ]
)

,member_ous_uid,obs_id_source,archive_row_count,spw_count,target_names,asdm_uids,ra_values,dec_values,region_count,spatial_resolutions,antenna_arrays_values,support_string_count,frequency_range_signature_count
9,uid://A001/X13e/Xc,ADFS_17,4,4,"(ADFS_17,)","(uid://A002/Xa95c04/X1e3e,)","(69.8077,)","(-54.4296,)",1,"(0.15279583881018002,)",(A007:DV09 A008:DV08 A011:DV04 A014:DV06 A015:DV18 A021:DA59 A058:DA46 A060:DA55 A069:DA53 A072:DV01 A076:DV13 A077:...,1,1
10,uid://A001/X13e/Xc,ADFS_27,4,4,"(ADFS_27,)","(uid://A002/Xa95c04/X1e3e,)","(69.2353,)","(-54.63740000000001,)",1,"(0.15291085196817553,)",(A007:DV09 A008:DV08 A011:DV04 A014:DV06 A015:DV18 A021:DA59 A058:DA46 A060:DA55 A069:DA53 A072:DV01 A076:DV13 A077:...,1,1
11,uid://A001/X13e/Xc,ADFS_31,4,4,"(ADFS_31,)","(uid://A002/Xa95c04/X1e3e,)","(71.2907,)","(-53.0028,)",1,"(0.15201040770219584,)",(A007:DV09 A008:DV08 A011:DV04 A014:DV06 A015:DV18 A021:DA59 A058:DA46 A060:DA55 A069:DA53 A072:DV01 A076:DV13 A077:...,1,1
12,uid://A001/X13e/Xc,ADFS_33,4,4,"(ADFS_33,)","(uid://A002/Xa95c04/X1e3e,)","(72.2904,)","(-53.3751,)",1,"(0.1520597366461261,)",(A007:DV09 A008:DV08 A011:DV04 A014:DV06 A015:DV18 A021:DA59 A058:DA46 A060:DA55 A069:DA53 A072:DV01 A076:DV13 A077:...,1,1
16,uid://A001/X21f/Xb,FMR2006_1,3,3,"(FMR2006_1,)","(uid://A002/Xa2d681/X1ad9,)","(279.4845833333333,)","(-6.875611111111111,)",1,"(0.42347479068793253,)",(A007:DV22 A011:DV04 A015:DV18 A017:DA57 A021:DA59 A029:DA49 A030:DA45 A031:DV08 A033:DV19 A035:DA52 A037:DV12 A038:...,1,1
17,uid://A001/X21f/Xb,FMR2006_10,3,3,"(FMR2006_10,)","(uid://A002/Xa2d681/X1ad9,)","(279.49804166666667,)","(-6.892222222222222,)",1,"(0.4232224192208648,)",(A007:DV22 A011:DV04 A015:DV18 A017:DA57 A021:DA59 A029:DA49 A030:DA45 A031:DV08 A033:DV19 A035:DA52 A037:DV12 A038:...,1,1
18,uid://A001/X21f/Xb,FMR2006_11,3,3,"(FMR2006_11,)","(uid://A002/Xa2d681/X1ad9,)","(279.46554166666664,)","(-6.863861111111111,)",1,"(0.42325638854076986,)",(A007:DV22 A011:DV04 A015:DV18 A017:DA57 A021:DA59 A029:DA49 A030:DA45 A031:DV08 A033:DV19 A035:DA52 A037:DV12 A038:...,1,1
19,uid://A001/X21f/Xb,FMR2006_12,3,3,"(FMR2006_12,)","(uid://A002/Xa2d681/X1ad9,)","(279.5137916666667,)","(-6.879194444444446,)",1,"(0.4231626540906479,)",(A007:DV22 A011:DV04 A015:DV18 A017:DA57 A021:DA59 A029:DA49 A030:DA45 A031:DV08 A033:DV19 A035:DA52 A037:DV12 A038:...,1,1
20,uid://A001/X21f/Xb,FMR2006_13,3,3,"(FMR2006_13,)","(uid://A002/Xa2d681/X1ad9,)","(279.4954583333333,)","(-6.875583333333333,)",1,"(0.4231398696405318,)",(A007:DV22 A011:DV04 A015:DV18 A017:DA57 A021:DA59 A029:DA49 A030:DA45 A031:DV08 A033:DV19 A035:DA52 A037:DV12 A038:...,1,1
21,uid://A001/X21f/Xb,FMR2006_14,3,3,"(FMR2006_14,)","(uid://A002/Xa2d681/X1ad9,)","(279.44854166666664,)","(-6.884,)",1,"(0.42309606048747156,)",(A007:DV22 A011:DV04 A015:DV18 A017:DA57 A021:DA59 A029:DA49 A030:DA45 A031:DV08 A033:DV19 A035:DA52 A037:DV12 A038:...,1,1


In [27]:
def count_dependency_violations(
    dataframe,
    determinant_columns,
    dependent_column,
):
    cardinality = (
        dataframe
        .groupby(
            determinant_columns,
            dropna=False,
        )[dependent_column]
        .nunique(dropna=False)
    )

    return int(
        (cardinality > 1).sum()
    )

    

In [28]:
dependency_checks = pd.DataFrame(
    [
        {
            "relationship":
                "(Member, source) -> target_name",
            "violations":
                count_dependency_violations(
                    relationship_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "target_name",
                ),
        },
        {
            "relationship":
                "(Member, source) -> ASDM",
            "violations":
                count_dependency_violations(
                    relationship_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "asdm_uid",
                ),
        },
        {
            "relationship":
                "(Member, source) -> frequency ranges",
            "violations":
                count_dependency_violations(
                    relationship_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "frequency_range_signature",
                ),
        },
        {
            "relationship":
                "(Member, source, SPW) -> frequency",
            "violations":
                count_dependency_violations(
                    relationship_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                        "spw_identifier",
                    ],
                    "frequency",
                ),
        },
        {
            "relationship":
                "ASDM -> antenna arrays",
            "violations":
                count_dependency_violations(
                    relationship_analysis_df,
                    ["asdm_uid"],
                    "antenna_arrays",
                ),
        },
        {
            "relationship":
                "ASDM -> frequency ranges",
            "violations":
                count_dependency_violations(
                    relationship_analysis_df,
                    ["asdm_uid"],
                    "frequency_range_signature",
                ),
        },
    ]
)

display(dependency_checks)

,relationship,violations
0,"(Member, source) -> target_name",0
1,"(Member, source) -> ASDM",0
2,"(Member, source) -> frequency ranges",0
3,"(Member, source, SPW) -> frequency",0
4,ASDM -> antenna arrays,0
5,ASDM -> frequency ranges,1


In [29]:
frequency_context_df = (
    relationship_analysis_df.loc[
        relationship_analysis_df[
            "member_ous_uid"
        ].isin(
            [
                "uid://A001/X5a4/Xe",
                "uid://A001/X62/X6",
            ]
        ),
        [
            "member_ous_uid",
            "obs_id_source",
            "target_name",
            "asdm_uid",
            "spw_identifier",
            "frequency",
            "bandwidth",
            "frequency_range_signature",
            "frequency_support",
            "antenna_arrays",
        ],
    ]
    .sort_values(
        [
            "member_ous_uid",
            "obs_id_source",
            "spw_identifier",
        ]
    )
)

display(frequency_context_df)

,member_ous_uid,obs_id_source,target_name,asdm_uid,spw_identifier,frequency,bandwidth,frequency_range_signature,frequency_support,antenna_arrays
153,uid://A001/X5a4/Xe,15470-5419c1,15470-5419c1,uid://A002/Xb1d975/X2cff,16,279.566658,1.250000e+08,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))","[277.07..279.05GHz,31250.00kHz,18.1mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.2mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
154,uid://A001/X5a4/Xe,15470-5419c1,15470-5419c1,uid://A002/Xb1d975/X2cff,18,278.058208,2.000000e+09,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))","[277.07..279.05GHz,31250.00kHz,18.1mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.2mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
155,uid://A001/X5a4/Xe,15470-5419c1,15470-5419c1,uid://A002/Xb1d975/X2cff,20,289.701763,2.500000e+08,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))","[277.07..279.05GHz,31250.00kHz,18.1mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.2mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
156,uid://A001/X5a4/Xe,15470-5419c1,15470-5419c1,uid://A002/Xb1d975/X2cff,22,288.200254,2.500000e+08,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))","[277.07..279.05GHz,31250.00kHz,18.1mJy/beam@10km/s,1.2mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.2mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
163,uid://A001/X5a4/Xe,15470-5419c3,15470-5419c3,uid://A002/Xb1d975/X2cff,16,279.566688,1.250000e+08,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))","[277.07..279.05GHz,31250.00kHz,18.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.9mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
164,uid://A001/X5a4/Xe,15470-5419c3,15470-5419c3,uid://A002/Xb1d975/X2cff,18,278.058228,2.000000e+09,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))","[277.07..279.05GHz,31250.00kHz,18.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.9mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
161,uid://A001/X5a4/Xe,15470-5419c3,15470-5419c3,uid://A002/Xb1d975/X2cff,20,289.701786,2.500000e+08,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))","[277.07..279.05GHz,31250.00kHz,18.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.9mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
162,uid://A001/X5a4/Xe,15470-5419c3,15470-5419c3,uid://A002/Xb1d975/X2cff,22,288.200267,2.500000e+08,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))","[277.07..279.05GHz,31250.00kHz,18.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.9mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
149,uid://A001/X5a4/Xe,15557-5215c2,15557-5215c2,uid://A002/Xb1d975/X2cff,16,279.565434,1.250000e+08,"((277.06, 279.05), (279.5, 279.63), (288.07, 288.32), (289.58, 289.83))","[277.06..279.05GHz,31250.00kHz,18.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY] U [279.50..279.63GHz,121.15kHz,19.9mJy...",J501:CM10 J502:CM02 J503:CM03 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N603:CM09 N604:CM11 N605:CM04 N606:CM06
150,uid://A001/X5a4/Xe,15557-5215c2,15557-5215c2,uid://A002/Xb1d975/X2cff,18,278.057069,2.000000e+09,"((277.06, 279.05), (279.5, 279.63), (288.07, 288.32), (289.58, 289.83))","[277.06..279.05G

In [30]:
for member_uid in [
    "uid://A001/X5a4/Xe",
    "uid://A001/X62/X6",
]:
    member_rows = (
        relationship_analysis_df.loc[
            relationship_analysis_df[
                "member_ous_uid"
            ] == member_uid
        ]
        .copy()
    )

    print("\nMember OUS:", member_uid)

    # Convert each nested frequency-range tuple to a plain string
    # before assigning a compact categorical ID.
    member_rows[
        "frequency_signature_text"
    ] = (
        member_rows[
            "frequency_range_signature"
        ]
        .map(repr)
    )

    signature_codes, _ = pd.factorize(
        member_rows[
            "frequency_signature_text"
        ],
        sort=False,
    )

    member_rows[
        "frequency_signature_id"
    ] = [
        (
            "missing"
            if code == -1
            else f"signature_{code + 1}"
        )
        for code in signature_codes
    ]

    print("Source-to-signature association:")

    source_signature_table = pd.crosstab(
        member_rows["obs_id_source"],
        member_rows["frequency_signature_id"],
    )

    display(source_signature_table)

    print("Signature definitions:")

    signature_lookup_df = (
        member_rows[
            [
                "frequency_signature_id",
                "frequency_range_signature",
            ]
        ]
        .drop_duplicates(
            subset="frequency_signature_id"
        )
        .sort_values(
            "frequency_signature_id"
        )
        .reset_index(drop=True)
    )

    display(signature_lookup_df)


Member OUS: uid://A001/X5a4/Xe
Source-to-signature association:


frequency_signature_id,signature_1,signature_2
obs_id_source,,
15470-5419c1,0,4
15470-5419c3,0,4
15557-5215c2,4,0
15557-5215c3,4,0


Signature definitions:


,frequency_signature_id,frequency_range_signature
0,signature_1,"((277.06, 279.05), (279.5, 279.63), (288.07, 288.32), (289.58, 289.83))"
1,signature_2,"((277.07, 279.05), (279.5, 279.63), (288.08, 288.33), (289.58, 289.83))"



Member OUS: uid://A001/X62/X6
Source-to-signature association:


frequency_signature_id,signature_1,signature_2
obs_id_source,,
HD 21997,0,4
HD21997,4,0


Signature definitions:


,frequency_signature_id,frequency_range_signature
0,signature_1,"((337.02, 339.01), (338.96, 340.94), (349.02, 351.01), (351.02, 353.01))"
1,signature_2,"((337.02, 339.0), (338.95, 340.94), (349.02, 351.0), (351.02, 353.0))"


## Step 6 — Quantify Differences Between Frequency-Range Signatures

Two Member OUS datasets contain more than one exact frequency-range
signature. This experiment measures the numerical differences between the
reported interval boundaries.

The purpose is not to define a scientific equivalence threshold. It is to
distinguish exact string inequality from the magnitude of the underlying
frequency difference.

In [31]:
from itertools import combinations

signature_context_df = (
    relationship_analysis_df[
        [
            "member_ous_uid",
            "obs_id_source",
            "frequency_range_signature",
        ]
    ]
    .copy()
)

signature_context_df[
    "frequency_signature_text"
] = (
    signature_context_df[
        "frequency_range_signature"
    ]
    .map(repr)
)

signature_context_df = (
    signature_context_df
    .drop_duplicates(
        subset=[
            "member_ous_uid",
            "obs_id_source",
            "frequency_signature_text",
        ]
    )
    .reset_index(drop=True)
)

signature_comparisons = []

for member_uid, member_group in (
    signature_context_df.groupby("member_ous_uid")
):
    signature_definitions = (
        member_group[
            [
                "frequency_signature_text",
                "frequency_range_signature",
            ]
        ]
        .drop_duplicates(
            subset="frequency_signature_text"
        )
        .reset_index(drop=True)
    )

    if len(signature_definitions) < 2:
        continue

    sources_by_signature = (
        member_group
        .groupby("frequency_signature_text")[
            "obs_id_source"
        ]
        .apply(
            lambda values: tuple(
                sorted(values.astype(str).unique())
            )
        )
        .to_dict()
    )

    for left_index, right_index in combinations(
        signature_definitions.index,
        2,
    ):
        left_signature = signature_definitions.loc[
            left_index,
            "frequency_range_signature",
        ]
        right_signature = signature_definitions.loc[
            right_index,
            "frequency_range_signature",
        ]

        if (
            left_signature is None
            or right_signature is None
            or len(left_signature) != len(right_signature)
        ):
            continue

        left_array = np.asarray(
            left_signature,
            dtype=float,
        )
        right_array = np.asarray(
            right_signature,
            dtype=float,
        )

        edge_differences_mhz = (
            np.abs(left_array - right_array)
            * 1000.0
        )

        left_centres = left_array.mean(axis=1)
        right_centres = right_array.mean(axis=1)

        centre_differences_mhz = (
            np.abs(left_centres - right_centres)
            * 1000.0
        )

        left_widths_mhz = (
            left_array[:, 1] - left_array[:, 0]
        ) * 1000.0

        right_widths_mhz = (
            right_array[:, 1] - right_array[:, 0]
        ) * 1000.0

        width_differences_mhz = np.abs(
            left_widths_mhz - right_widths_mhz
        )

        left_text = signature_definitions.loc[
            left_index,
            "frequency_signature_text",
        ]
        right_text = signature_definitions.loc[
            right_index,
            "frequency_signature_text",
        ]

        signature_comparisons.append(
            {
                "member_ous_uid": member_uid,
                "sources_signature_a":
                    sources_by_signature[left_text],
                "sources_signature_b":
                    sources_by_signature[right_text],
                "interval_count":
                    len(left_signature),
                "maximum_edge_difference_mhz":
                    edge_differences_mhz.max(),
                "maximum_centre_difference_mhz":
                    centre_differences_mhz.max(),
                "maximum_width_difference_mhz":
                    width_differences_mhz.max(),
                "mean_edge_difference_mhz":
                    edge_differences_mhz.mean(),
            }
        )

frequency_signature_comparison_df = pd.DataFrame(
    signature_comparisons
)

display(frequency_signature_comparison_df)

,member_ous_uid,sources_signature_a,sources_signature_b,interval_count,maximum_edge_difference_mhz,maximum_centre_difference_mhz,maximum_width_difference_mhz,mean_edge_difference_mhz
0,uid://A001/X5a4/Xe,"(15557-5215c2, 15557-5215c3)","(15470-5419c1, 15470-5419c3)",4,10.0,10.0,10.0,3.75
1,uid://A001/X62/X6,"(HD21997,)","(HD 21997,)",4,10.0,5.0,10.0,5.00


## Step 7 — Source Labels, Coordinates, and Physical Identity

Archive source labels are proposal-supplied strings and may contain formatting
differences. This experiment compares source labels and angular separations
within each multi-source Member OUS.

Name normalization and a small angular-separation threshold are used only to
identify cases for inspection. They are not treated as finalized source-
matching rules.

In [32]:
from astropy.coordinates import SkyCoord
import astropy.units as u


def normalize_source_label(value):
    if pd.isna(value):
        return None

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


source_identity_df = (
    relationship_analysis_df
    .groupby(
        [
            "member_ous_uid",
            "obs_id_source",
        ],
        dropna=False,
    )
    .agg(
        target_name=("target_name", "first"),
        s_ra=("s_ra", "first"),
        s_dec=("s_dec", "first"),
        s_region=("s_region", "first"),
        asdm_uid=("asdm_uid", "first"),
        spatial_resolution=(
            "spatial_resolution",
            "first",
        ),
        antenna_arrays=(
            "antenna_arrays",
            "first",
        ),
    )
    .reset_index()
)

source_identity_df[
    "normalized_source_label"
] = (
    source_identity_df[
        "obs_id_source"
    ]
    .apply(normalize_source_label)
)

source_pair_records = []

for member_uid, member_sources in (
    source_identity_df.groupby("member_ous_uid")
):
    member_sources = (
        member_sources
        .dropna(subset=["s_ra", "s_dec"])
        .reset_index(drop=True)
    )

    if len(member_sources) < 2:
        continue

    for left_index, right_index in combinations(
        member_sources.index,
        2,
    ):
        left_row = member_sources.loc[left_index]
        right_row = member_sources.loc[right_index]

        left_coord = SkyCoord(
            ra=float(left_row["s_ra"]) * u.deg,
            dec=float(left_row["s_dec"]) * u.deg,
            frame="icrs",
        )

        right_coord = SkyCoord(
            ra=float(right_row["s_ra"]) * u.deg,
            dec=float(right_row["s_dec"]) * u.deg,
            frame="icrs",
        )

        separation_arcsec = (
            left_coord
            .separation(right_coord)
            .arcsec
        )

        source_pair_records.append(
            {
                "member_ous_uid": member_uid,
                "source_a":
                    left_row["obs_id_source"],
                "source_b":
                    right_row["obs_id_source"],
                "normalized_names_equal":
                    (
                        left_row[
                            "normalized_source_label"
                        ]
                        ==
                        right_row[
                            "normalized_source_label"
                        ]
                    ),
                "separation_arcsec":
                    separation_arcsec,
                "same_asdm":
                    (
                        left_row["asdm_uid"]
                        == right_row["asdm_uid"]
                    ),
                "asdm_a": left_row["asdm_uid"],
                "asdm_b": right_row["asdm_uid"],
                "spatial_resolution_a":
                    left_row["spatial_resolution"],
                "spatial_resolution_b":
                    right_row["spatial_resolution"],
            }
        )

source_pair_df = pd.DataFrame(
    source_pair_records
)

interesting_source_pairs_df = (
    source_pair_df.loc[
        (
            source_pair_df[
                "normalized_names_equal"
            ]
        )
        |
        (
            source_pair_df[
                "separation_arcsec"
            ] <= 1.0
        )
    ]
    .sort_values(
        [
            "separation_arcsec",
            "member_ous_uid",
        ]
    )
    .reset_index(drop=True)
)

display(interesting_source_pairs_df)

,member_ous_uid,source_a,source_b,normalized_names_equal,separation_arcsec,same_asdm,asdm_a,asdm_b,spatial_resolution_a,spatial_resolution_b
0,uid://A001/X62/X6,HD 21997,HD21997,True,0.0,False,uid://A002/X2f146f/X700,uid://A002/X307fb7/X925,1.217369,1.11032


## Step 8 — Source Multiplicity, Mosaic State, and Footprint Geometry

Source multiplicity, mosaic status, and spatial-footprint geometry represent
different aspects of an observation. This experiment checks whether they are
associated in the current sample without assuming that one can be derived from
another.

In [33]:
def extract_region_geometry_type(value):
    if pd.isna(value):
        return "Missing"

    match = re.match(
        r"\s*([A-Za-z]+)",
        str(value),
    )

    if match is None:
        return "Unknown"

    return match.group(1).title()


relationship_analysis_df[
    "region_geometry_type"
] = (
    relationship_analysis_df[
        "s_region"
    ]
    .apply(extract_region_geometry_type)
)

member_geometry_summary = (
    relationship_analysis_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )
    .agg(
        is_mosaic=("is_mosaic", "first"),
        source_count=(
            "obs_id_source",
            "nunique",
        ),
        region_count=(
            "s_region",
            "nunique",
        ),
        geometry_types=(
            "region_geometry_type",
            unique_values_as_tuple,
        ),
        spw_count=(
            "spw_identifier",
            "nunique",
        ),
        archive_row_count=("obs_id", "size"),
    )
    .reset_index()
)

member_geometry_summary[
    "source_structure"
] = np.where(
    member_geometry_summary[
        "source_count"
    ] > 1,
    "Multiple sources",
    "Single source",
)

display(member_geometry_summary)

print("Mosaic state versus source multiplicity:")

display(
    pd.crosstab(
        member_geometry_summary["is_mosaic"],
        member_geometry_summary[
            "source_structure"
        ],
        margins=True,
    )
)

,member_ous_uid,is_mosaic,source_count,region_count,geometry_types,spw_count,archive_row_count,source_structure
0,uid://A001/X11e/Xb,F,1,1,"(Circle,)",4,4,Single source
1,uid://A001/X11f/X98,T,1,1,"(Polygon,)",4,4,Single source
2,uid://A001/X120/X17,T,1,1,"(Union,)",4,4,Single source
3,uid://A001/X122/X409,T,1,1,"(Polygon,)",4,4,Single source
4,uid://A001/X122/X411,T,1,1,"(Polygon,)",4,4,Single source
5,uid://A001/X125/X5,T,1,1,"(Polygon,)",4,4,Single source
6,uid://A001/X12d/X6,F,1,1,"(Circle,)",11,11,Single source
7,uid://A001/X13e/Xb8,F,1,1,"(Circle,)",4,4,Single source
8,uid://A001/X13e/Xba,F,1,1,"(Circle,)",4,4,Single source
9,uid://A001/X13e/Xc,F,4,4,"(Circle,)",4,16,Multiple sources


Mosaic state versus source multiplicity:


source_structure,Multiple sources,Single source,All
is_mosaic,,,
F,4,8,12
T,0,12,12
All,4,20,24


In [34]:
member_geometry_rows = (
    relationship_analysis_df[
        [
            "member_ous_uid",
            "is_mosaic",
            "region_geometry_type",
        ]
    ]
    .drop_duplicates()
)

print("Mosaic state versus s_region geometry:")

display(
    pd.crosstab(
        member_geometry_rows["is_mosaic"],
        member_geometry_rows[
            "region_geometry_type"
        ],
        margins=True,
    )
)

print("Multi-source Member OUS datasets:")

display(
    member_geometry_summary.loc[
        member_geometry_summary[
            "source_count"
        ] > 1
    ]
)

Mosaic state versus s_region geometry:


region_geometry_type,Circle,Polygon,Union,All
is_mosaic,,,,
F,10,2,0,12
T,0,10,2,12
All,10,12,2,24


Multi-source Member OUS datasets:


,member_ous_uid,is_mosaic,source_count,region_count,geometry_types,spw_count,archive_row_count,source_structure
9,uid://A001/X13e/Xc,F,4,4,"(Circle,)",4,16,Multiple sources
13,uid://A001/X21f/Xb,F,15,15,"(Circle,)",3,45,Multiple sources
16,uid://A001/X5a4/Xe,F,4,4,"(Polygon,)",4,16,Multiple sources
17,uid://A001/X62/X6,F,2,1,"(Circle,)",4,8,Multiple sources


## Step 9 — Validation with Recent Archive Datasets

The original stratified sample is dominated by observations from 2011–2016.
A second purposive sample is therefore drawn from proposal identifiers
beginning with `202`.

The recent sample is used to test whether the source–SPW row structure and
other preliminary relationships also occur in newer Archive records. It is
not intended to estimate Archive-wide frequencies or proportions.

In [37]:
recent_candidate_tables = []

for index, stratum in enumerate(
    sample_strata,
    start=1,
):
    print(
        f"Running stratum {index}/{len(sample_strata)}:",
        stratum["label"],
        flush=True,
    )

    recent_candidate_query = f"""
    SELECT DISTINCT TOP 12
        member_ous_uid,
        proposal_id,
        is_mosaic,
        obs_release_date
    FROM ivoa.obscore
    WHERE science_observation = 'T'
    AND member_ous_uid IS NOT NULL
    AND proposal_id LIKE '202%'
    AND frequency >= {stratum["frequency_min_ghz"]}
    AND frequency < {stratum["frequency_max_ghz"]}
    AND is_mosaic = '{stratum["is_mosaic"]}'
    ORDER BY obs_release_date DESC
    """

    recent_candidates = run_tap_query(
        recent_candidate_query,
        maxrec=100,
    )

    recent_candidates["sample_stratum"] = (
        stratum["label"]
    )

    recent_candidate_tables.append(
        recent_candidates
    )

    print(
        f"Finished: {len(recent_candidates)} candidates",
        flush=True,
    )

print("All recent candidate queries completed.")

Running stratum 1/8: band3_non_mosaic
Finished: 12 candidates
Running stratum 2/8: band3_mosaic
Finished: 12 candidates
Running stratum 3/8: band6_non_mosaic
Finished: 12 candidates
Running stratum 4/8: band6_mosaic
Finished: 12 candidates
Running stratum 5/8: band7_non_mosaic
Finished: 12 candidates
Running stratum 6/8: band7_mosaic
Finished: 12 candidates
Running stratum 7/8: band9_non_mosaic
Finished: 12 candidates
Running stratum 8/8: band9_mosaic
Finished: 12 candidates
All recent candidate queries completed.


In [39]:
print(
    "Completed query groups:",
    len(recent_candidate_tables),
)

assert len(recent_candidate_tables) > 0, (
    "recent_candidate_tables is empty."
)

recent_candidate_member_df = (
    pd.concat(
        recent_candidate_tables,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "sample_stratum",
            "member_ous_uid",
        ]
    )
    .reset_index(drop=True)
)

display(
    recent_candidate_member_df
    .groupby("sample_stratum")
    .size()
    .rename("recent_candidate_count")
    .to_frame()
)

Completed query groups: 8


,recent_candidate_count
sample_stratum,
band3_mosaic,12
band3_non_mosaic,12
band6_mosaic,12
band6_non_mosaic,12
band7_mosaic,12
band7_non_mosaic,12
band9_mosaic,12
band9_non_mosaic,12


In [40]:
recent_selected_groups = []

for stratum, group in (
    recent_candidate_member_df.groupby(
        "sample_stratum"
    )
):
    recent_selected_groups.append(
        group
        .sort_values(
            "obs_release_date",
            ascending=False,
        )
        .drop_duplicates(
            "member_ous_uid"
        )
        .head(3)
    )

recent_selected_member_df = (
    pd.concat(
        recent_selected_groups,
        ignore_index=True,
    )
    .drop_duplicates(
        "member_ous_uid"
    )
    .reset_index(drop=True)
)

display(recent_selected_member_df)

print(
    "Recent Member OUS datasets selected:",
    recent_selected_member_df[
        "member_ous_uid"
    ].nunique(),
)

,member_ous_uid,proposal_id,is_mosaic,obs_release_date,sample_stratum
0,uid://A001/X3833/X4b69,2025.1.00266.S,T,3000-01-01T00:00:00.000,band3_mosaic
1,uid://A001/X383d/X1274,2025.1.00342.S,T,3000-01-01T00:00:00.000,band3_mosaic
2,uid://A001/X383d/X1280,2025.1.00342.S,T,3000-01-01T00:00:00.000,band3_mosaic
3,uid://A001/X158f/X13e,2021.1.01543.S,F,3000-01-01T00:00:00.000,band3_non_mosaic
4,uid://A001/X15bd/X253,2021.1.01567.V,F,3000-01-01T00:00:00.000,band3_non_mosaic
5,uid://A001/X3667/X426,2023.1.00962.V,F,3000-01-01T00:00:00.000,band3_non_mosaic
6,uid://A001/X2d20/X2a46,2022.1.00403.S,T,3000-01-01T00:00:00.000,band6_mosaic
7,uid://A001/X3841/X164,2025.1.00114.S,T,3000-01-01T00:00:00.000,band6_mosaic
8,uid://A001/X3873/X544,2025.1.00269.S,T,3000-01-01T00:00:00.000,band6_mosaic
9,uid://A001/X15b8/X34a,2021.1.01156.V,F,3000-01-01T00:00:00.000,band6_non_mosaic


Recent Member OUS datasets selected: 24


In [41]:
recent_member_uids = (
    recent_selected_member_df[
        "member_ous_uid"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

recent_member_uid_sql = ",\n        ".join(
    f"'{member_uid}'"
    for member_uid in recent_member_uids
)

recent_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
        {recent_member_uid_sql}
)
"""

recent_count_df = run_tap_query(
    recent_count_query,
    maxrec=1,
)

recent_expected_row_count = int(
    recent_count_df[
        "total_rows"
    ].iloc[0]
)

recent_complete_query = f"""
SELECT
    {selected_columns_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
        {recent_member_uid_sql}
)
"""

recent_complete_member_df = run_tap_query(
    recent_complete_query,
    maxrec=max(
        recent_expected_row_count,
        1,
    ),
)

print(
    "Expected recent rows:",
    recent_expected_row_count,
)
print(
    "Retrieved recent rows:",
    len(recent_complete_member_df),
)
print(
    "Complete recent retrieval:",
    (
        len(recent_complete_member_df)
        == recent_expected_row_count
    ),
)

Expected recent rows: 153
Retrieved recent rows: 153
Complete recent retrieval: True


In [42]:
def build_relationship_summary(dataframe):
    parsed_df = (
        dataframe["obs_id"]
        .astype("string")
        .str.extract(obs_id_pattern)
    )

    analysis_df = pd.concat(
        [
            dataframe.reset_index(drop=True),
            parsed_df.reset_index(drop=True),
        ],
        axis=1,
    )

    analysis_df[
        "support_interval_count"
    ] = (
        analysis_df[
            "frequency_support"
        ]
        .apply(
            count_frequency_support_intervals
        )
    )

    summary_df = (
        analysis_df
        .groupby(
            "member_ous_uid",
            dropna=False,
        )
        .agg(
            archive_row_count=(
                "obs_id",
                "size",
            ),
            parsed_source_count=(
                "obs_id_source",
                "nunique",
            ),
            spw_identifier_count=(
                "spw_identifier",
                "nunique",
            ),
            support_interval_min=(
                "support_interval_count",
                "min",
            ),
            support_interval_max=(
                "support_interval_count",
                "max",
            ),
            asdm_count=(
                "asdm_uid",
                "nunique",
            ),
        )
        .reset_index()
    )

    summary_df[
        "expected_source_spw_rows"
    ] = (
        summary_df["parsed_source_count"]
        *
        summary_df["spw_identifier_count"]
    )

    summary_df[
        "rows_equal_source_spw_product"
    ] = (
        summary_df["archive_row_count"]
        ==
        summary_df[
            "expected_source_spw_rows"
        ]
    )

    summary_df[
        "spw_count_equals_support_intervals"
    ] = (
        summary_df[
            "spw_identifier_count"
        ]
        ==
        summary_df[
            "support_interval_min"
        ]
    )

    summary_df[
        "support_interval_count_consistent"
    ] = (
        summary_df[
            "support_interval_min"
        ]
        ==
        summary_df[
            "support_interval_max"
        ]
    )

    return analysis_df, summary_df


(
    recent_relationship_analysis_df,
    recent_member_structure_summary,
) = build_relationship_summary(
    recent_complete_member_df
)

display(recent_member_structure_summary)

print(
    "Unparsed recent obs_id rows:",
    recent_relationship_analysis_df[
        "spw_identifier"
    ].isna().sum(),
)

print(
    "Recent Members matching source × SPW:",
    recent_member_structure_summary[
        "rows_equal_source_spw_product"
    ].sum(),
    "/",
    len(recent_member_structure_summary),
)

print(
    "Recent Members matching SPW × support intervals:",
    recent_member_structure_summary[
        "spw_count_equals_support_intervals"
    ].sum(),
    "/",
    len(recent_member_structure_summary),
)

,member_ous_uid,archive_row_count,parsed_source_count,spw_identifier_count,support_interval_min,support_interval_max,asdm_count,expected_source_spw_rows,rows_equal_source_spw_product,spw_count_equals_support_intervals,support_interval_count_consistent
0,uid://A001/X158f/X13e,20,1,20,20,20,1,20,True,True,True
1,uid://A001/X15a1/Xe25,4,1,4,4,4,1,4,True,True,True
2,uid://A001/X15a1/Xe33,4,1,4,4,4,1,4,True,True,True
3,uid://A001/X15a1/Xe3b,4,1,4,4,4,1,4,True,True,True
4,uid://A001/X15b8/X34a,4,1,4,4,4,1,4,True,True,True
5,uid://A001/X15b8/X354,4,1,4,4,4,1,4,True,True,True
6,uid://A001/X15b8/X360,4,1,4,4,4,1,4,True,True,True
7,uid://A001/X15bd/X253,4,1,4,4,4,1,4,True,True,True
8,uid://A001/X2d20/X2a46,5,1,5,5,5,1,5,True,True,True
9,uid://A001/X3667/X426,4,1,4,4,4,1,4,True,True,True


Unparsed recent obs_id rows: 0
Recent Members matching source × SPW: 24 / 24
Recent Members matching SPW × support intervals: 24 / 24


In [43]:
recent_structure_issues_df = (
    recent_member_structure_summary.loc[
        (
            ~recent_member_structure_summary[
                "rows_equal_source_spw_product"
            ]
        )
        |
        (
            ~recent_member_structure_summary[
                "spw_count_equals_support_intervals"
            ]
        )
        |
        (
            ~recent_member_structure_summary[
                "support_interval_count_consistent"
            ]
        )
        |
        (
            recent_member_structure_summary[
                "asdm_count"
            ] > 1
        )
    ]
)

display(recent_structure_issues_df)

,member_ous_uid,archive_row_count,parsed_source_count,spw_identifier_count,support_interval_min,support_interval_max,asdm_count,expected_source_spw_rows,rows_equal_source_spw_product,spw_count_equals_support_intervals,support_interval_count_consistent


## Step 10 — Cross-Sample Summary and Data-Model Implications

The original stratified sample and the recent-cycle validation sample are now
compared. Relationships are classified as sample-supported, conditional,
rejected, or not yet tested.

A relationship observed in every sampled dataset is not automatically treated
as an Archive-wide schema constraint.

In [44]:
cross_sample_summary = pd.DataFrame(
    [
        {
            "sample": "Original stratified sample",
            "member_ous_count":
                len(member_structure_summary),
            "archive_row_count":
                int(
                    member_structure_summary[
                        "archive_row_count"
                    ].sum()
                ),
            "multi_source_members":
                int(
                    (
                        member_structure_summary[
                            "parsed_source_count"
                        ] > 1
                    ).sum()
                ),
            "source_spw_structure_matches":
                int(
                    member_structure_summary[
                        "rows_equal_source_spw_product"
                    ].sum()
                ),
            "spw_support_count_matches":
                int(
                    member_structure_summary[
                        "spw_count_equals_support_intervals"
                    ].sum()
                ),
            "members_with_multiple_asdms":
                int(
                    (
                        member_identifier_summary[
                            "asdm_count"
                        ] > 1
                    ).sum()
                ),
        },
        {
            "sample": "Recent-cycle validation sample",
            "member_ous_count":
                len(
                    recent_member_structure_summary
                ),
            "archive_row_count":
                int(
                    recent_member_structure_summary[
                        "archive_row_count"
                    ].sum()
                ),
            "multi_source_members":
                int(
                    (
                        recent_member_structure_summary[
                            "parsed_source_count"
                        ] > 1
                    ).sum()
                ),
            "source_spw_structure_matches":
                int(
                    recent_member_structure_summary[
                        "rows_equal_source_spw_product"
                    ].sum()
                ),
            "spw_support_count_matches":
                int(
                    recent_member_structure_summary[
                        "spw_count_equals_support_intervals"
                    ].sum()
                ),
            "members_with_multiple_asdms":
                int(
                    (
                        recent_member_structure_summary[
                            "asdm_count"
                        ] > 1
                    ).sum()
                ),
        },
    ]
)

display(cross_sample_summary)

,sample,member_ous_count,archive_row_count,multi_source_members,source_spw_structure_matches,spw_support_count_matches,members_with_multiple_asdms
0,Original stratified sample,24,193,4,24,24,1
1,Recent-cycle validation sample,24,153,1,24,24,0


In [45]:
recent_multi_source_uids = (
    recent_member_structure_summary.loc[
        recent_member_structure_summary[
            "parsed_source_count"
        ] > 1,
        "member_ous_uid",
    ]
)

for member_uid in recent_multi_source_uids:
    recent_member_rows = (
        recent_relationship_analysis_df.loc[
            recent_relationship_analysis_df[
                "member_ous_uid"
            ] == member_uid
        ]
    )

    print(
        "Recent multi-source Member:",
        member_uid,
    )

    display(
        pd.crosstab(
            recent_member_rows[
                "obs_id_source"
            ],
            recent_member_rows[
                "spw_identifier"
            ],
        )
    )

Recent multi-source Member: uid://A001/X3788/Xb60e


spw_identifier,21,23,25,27,29,31,33,35
obs_id_source,,,,,,,,
Flying_saucer,1,1,1,1,1,1,1,1
OphE-MM3,1,1,1,1,1,1,1,1
Oph_163131,1,1,1,1,1,1,1,1


## Step 11 — Recent-Sample Metadata Consistency

The original sample showed that some fields are stable at Member OUS level,
while source-related fields may contain multiple values within a Member.

This experiment repeats the field-cardinality and functional-dependency checks
on the recent-proposal sample. The purpose is to determine whether the
preliminary Source Context grouping is also useful for newer Archive records.

In [46]:
recent_member_field_cardinality = (
    recent_complete_member_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )[candidate_dataset_fields]
    .nunique(dropna=False)
)

recent_field_conflict_summary = (
    recent_member_field_cardinality
    .gt(1)
    .sum()
    .sort_values(ascending=False)
    .rename(
        "recent_members_with_multiple_values"
    )
    .to_frame()
)

display(recent_field_conflict_summary)

,recent_members_with_multiple_values
target_name,1
s_ra,1
s_dec,1
s_region,1
spatial_resolution,1
cont_sensitivity_bandwidth,1
proposal_id,0
group_ous_uid,0
asdm_uid,0
antenna_arrays,0


In [47]:
recent_conflicting_members = (
    recent_member_field_cardinality
    .gt(1)
    .any(axis=1)
)

display(
    recent_member_field_cardinality.loc[
        recent_conflicting_members
    ]
)

,proposal_id,group_ous_uid,asdm_uid,target_name,s_ra,s_dec,s_region,spatial_resolution,antenna_arrays,is_mosaic,cont_sensitivity_bandwidth,frequency_support,band_list,obs_release_date
member_ous_uid,,,,,,,,,,,,,,
uid://A001/X3788/Xb60e,1,1,1,3,3,3,3,3,1,1,3,1,1,1


In [48]:
recent_relationship_analysis_df[
    "frequency_range_signature"
] = (
    recent_relationship_analysis_df[
        "frequency_support"
    ]
    .apply(frequency_range_signature)
)

In [49]:
def build_dependency_check_table(
    dataframe,
    sample_name,
):
    checks = [
        {
            "relationship":
                "(Member, source) -> target_name",
            "determinants": [
                "member_ous_uid",
                "obs_id_source",
            ],
            "dependent": "target_name",
        },
        {
            "relationship":
                "(Member, source) -> ASDM",
            "determinants": [
                "member_ous_uid",
                "obs_id_source",
            ],
            "dependent": "asdm_uid",
        },
        {
            "relationship":
                "(Member, source) -> frequency ranges",
            "determinants": [
                "member_ous_uid",
                "obs_id_source",
            ],
            "dependent":
                "frequency_range_signature",
        },
        {
            "relationship":
                "(Member, source, SPW) -> frequency",
            "determinants": [
                "member_ous_uid",
                "obs_id_source",
                "spw_identifier",
            ],
            "dependent": "frequency",
        },
        {
            "relationship":
                "ASDM -> antenna arrays",
            "determinants": ["asdm_uid"],
            "dependent": "antenna_arrays",
        },
        {
            "relationship":
                "ASDM -> frequency ranges",
            "determinants": ["asdm_uid"],
            "dependent":
                "frequency_range_signature",
        },
        {
            "relationship":
                "ASDM -> Member OUS",
            "determinants": ["asdm_uid"],
            "dependent": "member_ous_uid",
        },
    ]

    records = []

    for check in checks:
        records.append(
            {
                "sample": sample_name,
                "relationship":
                    check["relationship"],
                "violations":
                    count_dependency_violations(
                        dataframe,
                        check["determinants"],
                        check["dependent"],
                    ),
            }
        )

    return pd.DataFrame(records)


original_dependency_check_table = (
    build_dependency_check_table(
        relationship_analysis_df,
        "Original sample",
    )
)

recent_dependency_check_table = (
    build_dependency_check_table(
        recent_relationship_analysis_df,
        "Recent sample",
    )
)

cross_sample_dependency_checks = pd.concat(
    [
        original_dependency_check_table,
        recent_dependency_check_table,
    ],
    ignore_index=True,
)

display(
    cross_sample_dependency_checks.pivot(
        index="relationship",
        columns="sample",
        values="violations",
    )
)

sample,Original sample,Recent sample
relationship,,
"(Member, source) -> ASDM",0,0
"(Member, source) -> frequency ranges",0,0
"(Member, source) -> target_name",0,0
"(Member, source, SPW) -> frequency",0,0
ASDM -> Member OUS,0,0
ASDM -> antenna arrays,0,0
ASDM -> frequency ranges,1,0


## Step 12 — Candidate Keys and Identifier Scope

This experiment evaluates possible keys for raw Archive records and tests the
scope of spectral-window identifiers.

A key that is unique in the current sample is treated only as a candidate key.
Its official Archive semantics must still be confirmed.

In [50]:
original_relationships_for_keys = (
    relationship_analysis_df.copy()
)

original_relationships_for_keys[
    "analysis_sample"
] = "Original"

recent_relationships_for_keys = (
    recent_relationship_analysis_df.copy()
)

recent_relationships_for_keys[
    "analysis_sample"
] = "Recent"

combined_relationship_df = pd.concat(
    [
        original_relationships_for_keys,
        recent_relationships_for_keys,
    ],
    ignore_index=True,
)

print(
    "Combined Archive rows:",
    len(combined_relationship_df),
)

print(
    "Combined Member OUS datasets:",
    combined_relationship_df[
        "member_ous_uid"
    ].nunique(),
)

Combined Archive rows: 346
Combined Member OUS datasets: 48


In [51]:
def evaluate_candidate_key(
    dataframe,
    key_name,
    key_columns,
):
    key_group_sizes = (
        dataframe
        .groupby(
            key_columns,
            dropna=False,
        )
        .size()
    )

    duplicate_groups = (
        key_group_sizes > 1
    )

    return {
        "candidate_key": key_name,
        "columns": ", ".join(key_columns),
        "total_rows": len(dataframe),
        "unique_key_count":
            len(key_group_sizes),
        "duplicate_key_groups":
            int(duplicate_groups.sum()),
        "rows_in_duplicate_groups":
            int(
                key_group_sizes.loc[
                    duplicate_groups
                ].sum()
            ),
        "is_unique_in_sample":
            bool(
                not duplicate_groups.any()
            ),
    }


candidate_key_results = pd.DataFrame(
    [
        evaluate_candidate_key(
            combined_relationship_df,
            "obs_id",
            ["obs_id"],
        ),
        evaluate_candidate_key(
            combined_relationship_df,
            "Member + source + SPW",
            [
                "member_ous_uid",
                "obs_id_source",
                "spw_identifier",
            ],
        ),
        evaluate_candidate_key(
            combined_relationship_df,
            "ASDM + source + SPW",
            [
                "asdm_uid",
                "obs_id_source",
                "spw_identifier",
            ],
        ),
        evaluate_candidate_key(
            combined_relationship_df,
            "Member + SPW",
            [
                "member_ous_uid",
                "spw_identifier",
            ],
        ),
    ]
)

display(candidate_key_results)

,candidate_key,columns,total_rows,unique_key_count,duplicate_key_groups,rows_in_duplicate_groups,is_unique_in_sample
0,obs_id,obs_id,346,346,0,0,True
1,Member + source + SPW,"member_ous_uid, obs_id_source, spw_identifier",346,346,0,0,True
2,ASDM + source + SPW,"asdm_uid, obs_id_source, spw_identifier",346,346,0,0,True
3,Member + SPW,"member_ous_uid, spw_identifier",346,260,23,109,False


In [52]:
identifier_scope_checks = pd.DataFrame(
    [
        {
            "relationship":
                "Member -> Proposal",
            "violations":
                count_dependency_violations(
                    combined_relationship_df,
                    ["member_ous_uid"],
                    "proposal_id",
                ),
        },
        {
            "relationship":
                "Member -> Group OUS",
            "violations":
                count_dependency_violations(
                    combined_relationship_df,
                    ["member_ous_uid"],
                    "group_ous_uid",
                ),
        },
        {
            "relationship":
                "(Member, SPW) -> frequency",
            "violations":
                count_dependency_violations(
                    combined_relationship_df,
                    [
                        "member_ous_uid",
                        "spw_identifier",
                    ],
                    "frequency",
                ),
        },
        {
            "relationship":
                "(Member, SPW) -> bandwidth",
            "violations":
                count_dependency_violations(
                    combined_relationship_df,
                    [
                        "member_ous_uid",
                        "spw_identifier",
                    ],
                    "bandwidth",
                ),
        },
        {
            "relationship":
                "(ASDM, SPW) -> frequency",
            "violations":
                count_dependency_violations(
                    combined_relationship_df,
                    [
                        "asdm_uid",
                        "spw_identifier",
                    ],
                    "frequency",
                ),
        },
        {
            "relationship":
                "ASDM -> Member",
            "violations":
                count_dependency_violations(
                    combined_relationship_df,
                    ["asdm_uid"],
                    "member_ous_uid",
                ),
        },
    ]
)

display(identifier_scope_checks)

,relationship,violations
0,Member -> Proposal,0
1,Member -> Group OUS,0
2,"(Member, SPW) -> frequency",23
3,"(Member, SPW) -> bandwidth",0
4,"(ASDM, SPW) -> frequency",19
5,ASDM -> Member,0


In [53]:
member_spw_frequency_spread = (
    combined_relationship_df
    .groupby(
        [
            "member_ous_uid",
            "spw_identifier",
        ],
        dropna=False,
    )
    .agg(
        source_count=(
            "obs_id_source",
            "nunique",
        ),
        frequency_value_count=(
            "frequency",
            "nunique",
        ),
        minimum_frequency_ghz=(
            "frequency",
            "min",
        ),
        maximum_frequency_ghz=(
            "frequency",
            "max",
        ),
    )
    .reset_index()
)

member_spw_frequency_spread[
    "frequency_spread_mhz"
] = (
    member_spw_frequency_spread[
        "maximum_frequency_ghz"
    ]
    -
    member_spw_frequency_spread[
        "minimum_frequency_ghz"
    ]
) * 1000.0

display(
    member_spw_frequency_spread.loc[
        (
            member_spw_frequency_spread[
                "source_count"
            ] > 1
        )
        &
        (
            member_spw_frequency_spread[
                "frequency_spread_mhz"
            ] > 0
        )
    ]
    .sort_values(
        "frequency_spread_mhz",
        ascending=False,
    )
)

,member_ous_uid,spw_identifier,source_count,frequency_value_count,minimum_frequency_ghz,maximum_frequency_ghz,frequency_spread_mhz
222,uid://A001/X62/X6,15,2,2,352.008881,352.014398,5.517846
221,uid://A001/X62/X6,13,2,2,350.008799,350.014286,5.486498
220,uid://A001/X62/X6,11,2,2,339.945890,339.951219,5.328780
223,uid://A001/X62/X6,9,2,2,338.008312,338.013610,5.298412
159,uid://A001/X3788/Xb60e,35,3,3,681.981094,681.982512,1.417845
157,uid://A001/X3788/Xb60e,31,3,3,679.981088,679.982502,1.413687
155,uid://A001/X3788/Xb60e,27,3,3,678.023082,678.024492,1.409616
153,uid://A001/X3788/Xb60e,23,3,3,676.023076,676.024481,1.405458
152,uid://A001/X3788/Xb60e,21,3,3,665.981045,665.982430,1.384581
154,uid://A001/X3788/Xb60e,25,3,3,663.981039,663.982419,1.380423


## Step 13 — Missing Values and Sentinel Values

The previous experiments focused on relationships between non-missing values.
This experiment measures field completeness and identifies placeholder values
that may require special handling.

Missingness is evaluated separately for the original and recent samples because
Archive conventions may differ between observing cycles.

In [54]:
completeness_fields = [
    "proposal_id",
    "group_ous_uid",
    "member_ous_uid",
    "obs_id",
    "asdm_uid",
    "target_name",
    "s_ra",
    "s_dec",
    "s_region",
    "frequency",
    "bandwidth",
    "frequency_support",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "antenna_arrays",
    "is_mosaic",
    "band_list",
    "obs_release_date",
]

In [55]:
missing_value_records = []

for sample_name, sample_df in [
    (
        "Original",
        relationship_analysis_df,
    ),
    (
        "Recent",
        recent_relationship_analysis_df,
    ),
]:
    for field_name in completeness_fields:
        field_series = sample_df[field_name]

        missing_count = int(
            field_series.isna().sum()
        )

        empty_string_count = int(
            field_series
            .astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
            .sum()
        )

        missing_value_records.append(
            {
                "sample": sample_name,
                "field": field_name,
                "row_count": len(sample_df),
                "missing_count":
                    missing_count,
                "empty_string_count":
                    empty_string_count,
                "missing_percentage":
                    (
                        100.0
                        * missing_count
                        / len(sample_df)
                    ),
            }
        )

missing_value_summary = pd.DataFrame(
    missing_value_records
)

display(
    missing_value_summary.loc[
        (
            missing_value_summary[
                "missing_count"
            ] > 0
        )
        |
        (
            missing_value_summary[
                "empty_string_count"
            ] > 0
        )
    ]
    .sort_values(
        [
            "field",
            "sample",
        ]
    )
)

,sample,field,row_count,missing_count,empty_string_count,missing_percentage
1,Original,group_ous_uid,193,0,24,0.0


In [56]:
release_date_diagnostics = []

for sample_name, sample_df in [
    (
        "Original",
        relationship_analysis_df,
    ),
    (
        "Recent",
        recent_relationship_analysis_df,
    ),
]:
    release_text = (
        sample_df[
            "obs_release_date"
        ]
        .astype("string")
    )

    release_date_diagnostics.append(
        {
            "sample": sample_name,
            "row_count": len(sample_df),
            "year_3000_rows":
                int(
                    release_text
                    .str.startswith("3000-")
                    .fillna(False)
                    .sum()
                ),
            "unique_release_dates":
                release_text.nunique(
                    dropna=False
                ),
        }
    )

release_date_diagnostic_df = pd.DataFrame(
    release_date_diagnostics
)

display(release_date_diagnostic_df)

,sample,row_count,year_3000_rows,unique_release_dates
0,Original,193,4,23
1,Recent,153,153,1


In [57]:
numeric_fields = [
    "s_ra",
    "s_dec",
    "frequency",
    "bandwidth",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
]

numeric_quality_records = []

for sample_name, sample_df in [
    (
        "Original",
        relationship_analysis_df,
    ),
    (
        "Recent",
        recent_relationship_analysis_df,
    ),
]:
    for field_name in numeric_fields:
        numeric_values = pd.to_numeric(
            sample_df[field_name],
            errors="coerce",
        )

        numeric_quality_records.append(
            {
                "sample": sample_name,
                "field": field_name,
                "zero_count":
                    int(
                        (
                            numeric_values == 0
                        ).sum()
                    ),
                "negative_count":
                    int(
                        (
                            numeric_values < 0
                        ).sum()
                    ),
                "non_finite_count":
                    int(
                        (
                            numeric_values.notna()
                            &
                            ~np.isfinite(
                                numeric_values
                            )
                        ).sum()
                    ),
            }
        )

numeric_quality_summary = pd.DataFrame(
    numeric_quality_records
)

display(
    numeric_quality_summary.loc[
        (
            numeric_quality_summary[
                [
                    "zero_count",
                    "negative_count",
                    "non_finite_count",
                ]
            ]
            .gt(0)
            .any(axis=1)
        )
    ]
)

,sample,field,zero_count,negative_count,non_finite_count
1,Original,s_dec,0,174,0
8,Recent,s_dec,0,79,0


## Step 14 — Proposal, Group OUS, Member OUS, and ASDM Hierarchy

The previous experiments began with selected Member OUS identifiers. They did
not retrieve every Member OUS belonging to the same proposal.

This experiment retrieves complete science-observation records for three
purposefully selected proposals and examines the cardinalities between
Proposal, Group OUS, Member OUS, and ASDM identifiers.

In [58]:
hierarchy_proposal_ids = [
    "2013.1.01200.S",
    "2013.1.00126.S",
    "2024.1.00928.S",
]

hierarchy_proposal_sql = ",\n        ".join(
    f"'{proposal_id}'"
    for proposal_id in hierarchy_proposal_ids
)

print(hierarchy_proposal_sql)

'2013.1.01200.S',
        '2013.1.00126.S',
        '2024.1.00928.S'


In [59]:
hierarchy_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND proposal_id IN (
        {hierarchy_proposal_sql}
)
"""

hierarchy_count_df = run_tap_query(
    hierarchy_count_query,
    maxrec=1,
)

hierarchy_expected_rows = int(
    hierarchy_count_df[
        "total_rows"
    ].iloc[0]
)

print(
    "Expected hierarchy rows:",
    hierarchy_expected_rows,
)

Expected hierarchy rows: 226


In [60]:
hierarchy_query = f"""
SELECT
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    asdm_uid,
    obs_id,
    target_name,
    is_mosaic,
    band_list,
    frequency,
    frequency_support
FROM ivoa.obscore
WHERE science_observation = 'T'
AND proposal_id IN (
        {hierarchy_proposal_sql}
)
"""

hierarchy_df = run_tap_query(
    hierarchy_query,
    maxrec=max(
        hierarchy_expected_rows,
        1,
    ),
)

print(
    "Expected hierarchy rows:",
    hierarchy_expected_rows,
)
print(
    "Retrieved hierarchy rows:",
    len(hierarchy_df),
)
print(
    "Complete retrieval:",
    (
        len(hierarchy_df)
        == hierarchy_expected_rows
    ),
)

Expected hierarchy rows: 226
Retrieved hierarchy rows: 226
Complete retrieval: True


In [61]:
proposal_hierarchy_summary = (
    hierarchy_df
    .groupby(
        "proposal_id",
        dropna=False,
    )
    .agg(
        archive_row_count=(
            "obs_id",
            "size",
        ),
        group_ous_count=(
            "group_ous_uid",
            "nunique",
        ),
        member_ous_count=(
            "member_ous_uid",
            "nunique",
        ),
        asdm_count=(
            "asdm_uid",
            "nunique",
        ),
        target_name_count=(
            "target_name",
            "nunique",
        ),
        mosaic_state_count=(
            "is_mosaic",
            "nunique",
        ),
        band_count=(
            "band_list",
            "nunique",
        ),
    )
    .reset_index()
)

display(proposal_hierarchy_summary)

,proposal_id,archive_row_count,group_ous_count,member_ous_count,asdm_count,target_name_count,mosaic_state_count,band_count
0,2013.1.00126.S,21,3,5,5,1,1,1
1,2013.1.01200.S,45,1,1,1,15,1,1
2,2024.1.00928.S,160,10,16,16,6,1,3


In [62]:
group_hierarchy_summary = (
    hierarchy_df
    .groupby(
        [
            "proposal_id",
            "group_ous_uid",
        ],
        dropna=False,
    )
    .agg(
        member_ous_count=(
            "member_ous_uid",
            "nunique",
        ),
        asdm_count=(
            "asdm_uid",
            "nunique",
        ),
        target_name_count=(
            "target_name",
            "nunique",
        ),
        archive_row_count=(
            "obs_id",
            "size",
        ),
    )
    .reset_index()
)

display(group_hierarchy_summary)

,proposal_id,group_ous_uid,member_ous_count,asdm_count,target_name_count,archive_row_count
0,2013.1.00126.S,uid://A001/X122/X406,2,2,1,8
1,2013.1.00126.S,uid://A001/X122/X40e,1,1,1,4
2,2013.1.00126.S,uid://A001/X122/X416,2,2,1,9
3,2013.1.01200.S,uid://A001/X21f/Xa,1,1,15,45
4,2024.1.00928.S,uid://A001/X3788/Xb5c5,1,1,1,4
5,2024.1.00928.S,uid://A001/X3788/Xb5ca,1,1,1,8
6,2024.1.00928.S,uid://A001/X3788/Xb5cf,1,1,1,4
7,2024.1.00928.S,uid://A001/X3788/Xb5d4,2,2,3,24
8,2024.1.00928.S,uid://A001/X3788/Xb5f4,2,2,1,16
9,2024.1.00928.S,uid://A001/X3788/Xb5fb,2,2,1,16


In [63]:
member_hierarchy_summary = (
    hierarchy_df
    .groupby(
        [
            "proposal_id",
            "group_ous_uid",
            "member_ous_uid",
        ],
        dropna=False,
    )
    .agg(
        asdm_count=(
            "asdm_uid",
            "nunique",
        ),
        target_name_count=(
            "target_name",
            "nunique",
        ),
        archive_row_count=(
            "obs_id",
            "size",
        ),
        mosaic_state_count=(
            "is_mosaic",
            "nunique",
        ),
        band_count=(
            "band_list",
            "nunique",
        ),
    )
    .reset_index()
)

display(member_hierarchy_summary)

,proposal_id,group_ous_uid,member_ous_uid,asdm_count,target_name_count,archive_row_count,mosaic_state_count,band_count
0,2013.1.00126.S,uid://A001/X122/X406,uid://A001/X122/X407,1,1,4,1,1
1,2013.1.00126.S,uid://A001/X122/X406,uid://A001/X122/X409,1,1,4,1,1
2,2013.1.00126.S,uid://A001/X122/X40e,uid://A001/X122/X411,1,1,4,1,1
3,2013.1.00126.S,uid://A001/X122/X416,uid://A001/X122/X417,1,1,4,1,1
4,2013.1.00126.S,uid://A001/X122/X416,uid://A001/X122/X419,1,1,5,1,1
5,2013.1.01200.S,uid://A001/X21f/Xa,uid://A001/X21f/Xb,1,15,45,1,1
6,2024.1.00928.S,uid://A001/X3788/Xb5c5,uid://A001/X3788/Xb5c6,1,1,4,1,1
7,2024.1.00928.S,uid://A001/X3788/Xb5ca,uid://A001/X3788/Xb5cb,1,1,8,1,1
8,2024.1.00928.S,uid://A001/X3788/Xb5cf,uid://A001/X3788/Xb5d0,1,1,4,1,1
9,2024.1.00928.S,uid://A001/X3788/Xb5d4,uid://A001/X3788/Xb5d5,1,3,12,1,1


In [64]:
hierarchy_dependency_checks = pd.DataFrame(
    [
        {
            "relationship":
                "Member OUS -> Group OUS",
            "violations":
                count_dependency_violations(
                    hierarchy_df,
                    ["member_ous_uid"],
                    "group_ous_uid",
                ),
        },
        {
            "relationship":
                "Member OUS -> Proposal",
            "violations":
                count_dependency_violations(
                    hierarchy_df,
                    ["member_ous_uid"],
                    "proposal_id",
                ),
        },
        {
            "relationship":
                "Group OUS -> Proposal",
            "violations":
                count_dependency_violations(
                    hierarchy_df,
                    ["group_ous_uid"],
                    "proposal_id",
                ),
        },
        {
            "relationship":
                "ASDM -> Member OUS",
            "violations":
                count_dependency_violations(
                    hierarchy_df,
                    ["asdm_uid"],
                    "member_ous_uid",
                ),
        },
    ]
)

display(hierarchy_dependency_checks)

,relationship,violations
0,Member OUS -> Group OUS,0
1,Member OUS -> Proposal,0
2,Group OUS -> Proposal,0
3,ASDM -> Member OUS,0


## Step 15 — Targeted Search for Structural Counterexamples

A larger recent candidate pool is inspected for counterexamples to the
preliminary relationships.

This is a targeted robustness test rather than a statistically representative
Archive survey. Any discovered counterexample will be used to expand the data
model rather than discarded as an outlier.

In [65]:
extended_recent_member_uids = (
    recent_candidate_member_df[
        "member_ous_uid"
    ]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

print(
    "Extended recent Member count:",
    len(extended_recent_member_uids),
)

Extended recent Member count: 96


In [66]:
extended_recent_uid_sql = ",\n        ".join(
    f"'{member_uid}'"
    for member_uid in (
        extended_recent_member_uids
    )
)

extended_recent_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
        {extended_recent_uid_sql}
)
"""

extended_recent_count_df = run_tap_query(
    extended_recent_count_query,
    maxrec=1,
)

extended_recent_expected_rows = int(
    extended_recent_count_df[
        "total_rows"
    ].iloc[0]
)

print(
    "Expected extended recent rows:",
    extended_recent_expected_rows,
)

Expected extended recent rows: 1226


In [67]:
extended_recent_query = f"""
SELECT
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    obs_id,
    obs_publisher_did,
    t_min,
    t_max, 
    asdm_uid,
    target_name,
    s_ra,
    s_dec,
    s_region,
    frequency,
    bandwidth,
    frequency_support,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    is_mosaic,
    band_list,
    obs_release_date
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
        {extended_recent_uid_sql}
)
"""

extended_recent_df = run_tap_query(
    extended_recent_query,
    maxrec=max(
        extended_recent_expected_rows,
        1,
    ),
)

print(
    "Retrieved extended rows:",
    len(extended_recent_df),
)
print(
    "Complete extended retrieval:",
    (
        len(extended_recent_df)
        == extended_recent_expected_rows
    ),
)

Retrieved extended rows: 1226
Complete extended retrieval: True


In [68]:
extended_parsed_obs_id_df = (
    extended_recent_df[
        "obs_id"
    ]
    .astype("string")
    .str.extract(obs_id_pattern)
)

extended_recent_analysis_df = pd.concat(
    [
        extended_recent_df.reset_index(
            drop=True
        ),
        extended_parsed_obs_id_df.reset_index(
            drop=True
        ),
    ],
    axis=1,
)

extended_recent_analysis_df[
    "frequency_range_signature"
] = (
    extended_recent_analysis_df[
        "frequency_support"
    ]
    .apply(frequency_range_signature)
)

print(
    "Unparsed extended obs_id rows:",
    extended_recent_analysis_df[
        "spw_identifier"
    ].isna().sum(),
)

Unparsed extended obs_id rows: 0


In [69]:
extended_member_summary = (
    extended_recent_analysis_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )
    .agg(
        proposal_id=(
            "proposal_id",
            "first",
        ),
        is_mosaic=(
            "is_mosaic",
            "first",
        ),
        archive_row_count=(
            "obs_id",
            "size",
        ),
        source_count=(
            "obs_id_source",
            "nunique",
        ),
        spw_count=(
            "spw_identifier",
            "nunique",
        ),
        asdm_count=(
            "asdm_uid",
            "nunique",
        ),
        target_name_count=(
            "target_name",
            "nunique",
        ),
        support_string_count=(
            "frequency_support",
            "nunique",
        ),
        frequency_signature_count=(
            "frequency_range_signature",
            "nunique",
        ),
        region_count=(
            "s_region",
            "nunique",
        ),
    )
    .reset_index()
)

extended_member_summary[
    "expected_source_spw_rows"
] = (
    extended_member_summary[
        "source_count"
    ]
    *
    extended_member_summary[
        "spw_count"
    ]
)

extended_member_summary[
    "matches_source_spw_structure"
] = (
    extended_member_summary[
        "archive_row_count"
    ]
    ==
    extended_member_summary[
        "expected_source_spw_rows"
    ]
)

In [70]:
print("Members with multiple ASDMs:")

display(
    extended_member_summary.loc[
        extended_member_summary[
            "asdm_count"
        ] > 1
    ]
)

print("Mosaic Members with multiple sources:")

display(
    extended_member_summary.loc[
        (
            extended_member_summary[
                "is_mosaic"
            ] == "T"
        )
        &
        (
            extended_member_summary[
                "source_count"
            ] > 1
        )
    ]
)

print("Members with multiple frequency signatures:")

display(
    extended_member_summary.loc[
        extended_member_summary[
            "frequency_signature_count"
        ] > 1
    ]
)

print("Members not matching source × SPW structure:")

display(
    extended_member_summary.loc[
        ~extended_member_summary[
            "matches_source_spw_structure"
        ]
    ]
)

Members with multiple ASDMs:


,member_ous_uid,proposal_id,is_mosaic,archive_row_count,source_count,spw_count,asdm_count,target_name_count,support_string_count,frequency_signature_count,region_count,expected_source_spw_rows,matches_source_spw_structure
36,uid://A001/X3833/X1022,2025.1.01276.S,T,8,2,4,2,2,2,1,2,8,True


Mosaic Members with multiple sources:


,member_ous_uid,proposal_id,is_mosaic,archive_row_count,source_count,spw_count,asdm_count,target_name_count,support_string_count,frequency_signature_count,region_count,expected_source_spw_rows,matches_source_spw_structure
36,uid://A001/X3833/X1022,2025.1.01276.S,T,8,2,4,2,2,2,1,2,8,True
44,uid://A001/X3833/X22b9,2025.1.00883.S,T,8,2,4,1,2,2,2,2,8,True
53,uid://A001/X3833/X2768,2025.1.00775.S,T,20,2,10,1,2,2,2,2,20,True
65,uid://A001/X3833/X4d0,2025.1.01547.S,T,112,8,14,1,8,2,1,8,112,True
76,uid://A001/X383d/X45a,2025.1.00576.L,T,8,2,4,1,2,2,1,2,8,True


Members with multiple frequency signatures:


,member_ous_uid,proposal_id,is_mosaic,archive_row_count,source_count,spw_count,asdm_count,target_name_count,support_string_count,frequency_signature_count,region_count,expected_source_spw_rows,matches_source_spw_structure
22,uid://A001/X3788/X7cc8,2024.1.01353.S,F,24,3,8,1,3,2,2,3,24,True
27,uid://A001/X3788/Xc584,2024.1.01279.S,F,256,16,16,1,16,4,3,16,256,True
35,uid://A001/X3819/X186,2024.1.00257.S,F,144,36,4,1,36,7,2,36,144,True
44,uid://A001/X3833/X22b9,2025.1.00883.S,T,8,2,4,1,2,2,2,2,8,True
53,uid://A001/X3833/X2768,2025.1.00775.S,T,20,2,10,1,2,2,2,2,20,True
67,uid://A001/X3833/X57f6,2025.1.00115.S,F,24,3,8,1,3,3,3,3,24,True


Members not matching source × SPW structure:


,member_ous_uid,proposal_id,is_mosaic,archive_row_count,source_count,spw_count,asdm_count,target_name_count,support_string_count,frequency_signature_count,region_count,expected_source_spw_rows,matches_source_spw_structure


In [71]:
extended_candidate_key_results = pd.DataFrame(
    [
        evaluate_candidate_key(
            extended_recent_analysis_df,
            "obs_id",
            ["obs_id"],
        ),
        evaluate_candidate_key(
            extended_recent_analysis_df,
            "Member + source + SPW",
            [
                "member_ous_uid",
                "obs_id_source",
                "spw_identifier",
            ],
        ),
    ]
)

display(extended_candidate_key_results)

,candidate_key,columns,total_rows,unique_key_count,duplicate_key_groups,rows_in_duplicate_groups,is_unique_in_sample
0,obs_id,obs_id,1226,1226,0,0,True
1,Member + source + SPW,"member_ous_uid, obs_id_source, spw_identifier",1226,1226,0,0,True


In [72]:
schema_query = """
SELECT
    column_name,
    datatype,
    arraysize,
    unit,
    ucd,
    utype,
    description
FROM TAP_SCHEMA.columns
WHERE table_name = 'ivoa.obscore'
"""

schema_result = service.search(schema_query)
schema_df = schema_result.to_table().to_pandas()

schema_df.sort_values("column_name").reset_index(drop=True)

,column_name,datatype,arraysize,unit,ucd,utype,description
0,access_estsize,int,,kbyte,phys.size;meta.file,obscore:Access.Size,Estimated size of datasets in kilobytes
1,access_format,char,9,,meta.code.mime,obscore:Access.Format,Content format of the data
2,access_url,char,72*,,meta.ref.url,obscore:Access.Reference,URL to download the data
3,antenna_arrays,char,660*,,meta.code.member;instr.setup,,"Blank-separated list of Pad:Antenna pairs, i.e., A109:DV09 J504:DV02 J505:DV05 for antennas DV09, DV02 and DV05 sitt..."
4,asdm_uid,char,32*,,meta.id,,UID of the ASDM containing this Field.
...,...,...,...,...,...,...,...
68,t_resolution,double,,s,time.resolution,obscore:Char.TimeAxis.Resolution.refval.value,typical temporal resolution
69,t_xel,int,,,meta.number,obscore:Char.TimeAxis.numBins,Number of elements along the time axis
70,target_name,char,256*,,meta.id;src,obscore:Target.Name,name of intended target
71,type,char,16*,,,,Type flags.


In [73]:
important_columns = [
    "obs_id",
    "obs_publisher_did",
    "proposal_id",
    "group_ous_uid",
    "member_ous_uid",
    "asdm_uid",
    "target_name",
    "s_ra",
    "s_dec",
    "s_region",
    "frequency",
    "bandwidth",
    "frequency_support",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
]

schema_df[
    schema_df["column_name"].isin(important_columns)
].sort_values("column_name")

,column_name,datatype,arraysize,unit,ucd,utype,description
31,asdm_uid,char,32*,,meta.id,,UID of the ASDM containing this Field.
54,bandwidth,double,,Hz,em.freq;instr.bandpass,obscore:Char.SpectralAxis.Coverage.Bounds.Extent,Total Bandwidth
27,cont_sensitivity_bandwidth,double,,mJy/beam,,,"Estimated noise in the aggregated continuum bandwidth. Note this is an indication only, it does not include the effe..."
60,frequency,double,,GHz,em.freq;obs;meta.main,obscore:Char.SpectralAxis.Coverage.Location.refval,Observed (tuned) reference frequency on the sky.
59,frequency_support,char,4000*,GHz,em.freq;obs;meta.main,obscore:Char.SpectralAxis.Coverage.Location.support,All frequency ranges used by the field
29,group_ous_uid,char,64*,,,,Group OUS ID
30,member_ous_uid,char,64*,,,,Member OUS ID
4,obs_id,char,64*,,meta.id,obscore:DataID.observationID,internal dataset identifier
0,obs_publisher_did,char,33*,,meta.ref.ivoid,obscore:Curation.PublisherDID,publisher dataset identifier
48,proposal_id,char,64*,,meta.id;obs.proposal,obscore:Provenance.Proposal.identifier,Identifier of proposal to which NO observation belongs.


In [74]:
identifier_columns = [
    "proposal_id",
    "member_ous_uid",
    "asdm_uid",
    "obs_id",
    "obs_publisher_did",
    "target_name",
]

extended_recent_df[identifier_columns].head()

,proposal_id,member_ous_uid,asdm_uid,obs_id,obs_publisher_did,target_name
0,2021.1.01123.L,uid://A001/X15a1/Xe3b,uid://A002/Xf1948b/X6406,uid://A001/X15a1/Xe3b.source.HD34282.spw.16,ADS/JAO.ALMA#2021.1.01123.L,HD34282
1,2021.1.01123.L,uid://A001/X15a1/Xe3b,uid://A002/Xf1948b/X6406,uid://A001/X15a1/Xe3b.source.HD34282.spw.18,ADS/JAO.ALMA#2021.1.01123.L,HD34282
2,2021.1.01123.L,uid://A001/X15a1/Xe3b,uid://A002/Xf1948b/X6406,uid://A001/X15a1/Xe3b.source.HD34282.spw.20,ADS/JAO.ALMA#2021.1.01123.L,HD34282
3,2021.1.01123.L,uid://A001/X15a1/Xe3b,uid://A002/Xf1948b/X6406,uid://A001/X15a1/Xe3b.source.HD34282.spw.22,ADS/JAO.ALMA#2021.1.01123.L,HD34282
4,2021.1.01156.V,uid://A001/X15b8/X354,uid://A002/Xf656ba/X990,uid://A001/X15b8/X354.source.NGC4261.spw.18,ADS/JAO.ALMA#2021.1.01156.V,NGC4261


In [75]:
identifier_summary = pd.DataFrame({
    "column": identifier_columns,
    "non_null_rows": [
        extended_recent_df[column].notna().sum()
        for column in identifier_columns
    ],
    "unique_values": [
        extended_recent_df[column].nunique(dropna=True)
        for column in identifier_columns
    ],
})

identifier_summary

,column,non_null_rows,unique_values
0,proposal_id,1226,59
1,member_ous_uid,1226,96
2,asdm_uid,1226,97
3,obs_id,1226,1226
4,obs_publisher_did,1226,59
5,target_name,1226,165


In [76]:
(
    extended_recent_df
    .groupby("obs_publisher_did", dropna=False)
    .agg(
        row_count=("obs_id", "size"),
        obs_id_count=("obs_id", "nunique"),
        member_count=("member_ous_uid", "nunique"),
        proposal_count=("proposal_id", "nunique"),
    )
    .sort_values("row_count", ascending=False)
    .head(20)
)

,row_count,obs_id_count,member_count,proposal_count
obs_publisher_did,,,,
ADS/JAO.ALMA#2024.1.01279.S,256,256,1,1
ADS/JAO.ALMA#2024.1.00257.S,144,144,1,1
ADS/JAO.ALMA#2025.1.01547.S,112,112,1,1
ADS/JAO.ALMA#2023.1.00607.S,80,80,1,1
ADS/JAO.ALMA#2025.1.00115.S,64,64,3,1
ADS/JAO.ALMA#2025.1.00791.S,56,56,7,1
ADS/JAO.ALMA#2024.1.01353.S,48,48,2,1
ADS/JAO.ALMA#2024.1.00928.S,32,32,2,1
ADS/JAO.ALMA#2025.1.00576.L,24,24,3,1


In [78]:
extended_parsed_obs_id_df = (
    extended_recent_df["obs_id"]
    .astype("string")
    .str.extract(obs_id_pattern)
)

extended_recent_analysis_df = pd.concat(
    [
        extended_recent_df.reset_index(drop=True),
        extended_parsed_obs_id_df.reset_index(drop=True),
    ],
    axis=1,
)

extended_recent_analysis_df[
    "frequency_range_signature"
] = (
    extended_recent_analysis_df[
        "frequency_support"
    ]
    .apply(frequency_range_signature)
)

print(
    "Unparsed extended obs_id rows:",
    extended_recent_analysis_df[
        "spw_identifier"
    ].isna().sum(),
)

Unparsed extended obs_id rows: 0


In [79]:
multi_asdm_required_columns = [
    "member_ous_uid",
    "asdm_uid",
    "obs_id_source",
    "spw_identifier",
    "frequency",
    "bandwidth",
    "antenna_arrays",
    "t_min",
    "t_max",
    "frequency_support",
]

missing_columns = [
    column
    for column in multi_asdm_required_columns
    if column not in extended_recent_analysis_df.columns
]

print("Missing columns:", missing_columns)

Missing columns: []


In [80]:
multi_asdm_uid = "uid://A001/X3833/X1022"

multi_asdm_case = (
    extended_recent_analysis_df.loc[
        extended_recent_analysis_df[
            "member_ous_uid"
        ] == multi_asdm_uid,
        multi_asdm_required_columns,
    ]
    .sort_values(
        [
            "asdm_uid",
            "obs_id_source",
            "spw_identifier",
        ]
    )
    .reset_index(drop=True)
)

print("Archive rows:", len(multi_asdm_case))
print(
    "ASDM count:",
    multi_asdm_case["asdm_uid"].nunique(),
)
print(
    "Source-context count:",
    multi_asdm_case["obs_id_source"].nunique(),
)
print(
    "SPW count:",
    multi_asdm_case["spw_identifier"].nunique(),
)

display(multi_asdm_case)

Archive rows: 8
ASDM count: 2
Source-context count: 2
SPW count: 4


,member_ous_uid,asdm_uid,obs_id_source,spw_identifier,frequency,bandwidth,antenna_arrays,t_min,t_max,frequency_support
0,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,24,345.382156,1.875000e+09,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N604:CM11 N605:CM02,61161.618347,61247.324103,"[344.45..346.32GHz,7812.01kHz,24.6mJy/beam@10km/s,1.9mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,24.5mJy..."
1,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,26,347.257122,1.875000e+09,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N604:CM11 N605:CM02,61161.618347,61247.324103,"[344.45..346.32GHz,7812.01kHz,24.6mJy/beam@10km/s,1.9mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,24.5mJy..."
2,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,28,357.611466,1.875000e+09,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N604:CM11 N605:CM02,61161.618347,61247.324103,"[344.45..346.32GHz,7812.01kHz,24.6mJy/beam@10km/s,1.9mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,24.5mJy..."
3,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,30,359.309397,1.875000e+09,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N602:CM01 N604:CM11 N605:CM02,61161.618347,61247.324103,"[344.45..346.32GHz,7812.01kHz,24.6mJy/beam@10km/s,1.9mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,24.5mJy..."
4,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,24,345.382200,1.875000e+09,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N604:CM11 N605:CM02,61164.618683,61247.350143,"[344.45..346.32GHz,7812.01kHz,21.8mJy/beam@10km/s,1.7mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,21.7mJy..."
5,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,26,347.257166,1.875000e+09,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N604:CM11 N605:CM02,61164.618683,61247.350143,"[344.45..346.32GHz,7812.01kHz,21.8mJy/beam@10km/s,1.7mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,21.7mJy..."
6,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,28,357.611513,1.875000e+09,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N604:CM11 N605:CM02,61164.618683,61247.350143,"[344.45..346.32GHz,7812.01kHz,21.8mJy/beam@10km/s,1.7mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,21.7mJy..."
7,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,30,359.309447,1.875000e+09,J501:CM10 J502:CM03 J503:CM04 J504:CM12 J505:CM08 J506:CM05 N601:CM07 N604:CM11 N605:CM02,61164.618683,61247.350143,"[344.45..346.32GHz,7812.01kHz,21.8mJy/beam@10km/s,1.7mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,21.7mJy..."


In [82]:
def extract_region_geometry_type(value):
    if pd.isna(value):
        return "Missing"

    match = re.match(
        r"\s*([A-Za-z]+)",
        str(value),
    )

    if match is None:
        return "Unknown"

    return match.group(1).title()


extended_spatial_analysis_df = (
    extended_recent_analysis_df.copy()
)

extended_spatial_analysis_df[
    "region_geometry_type"
] = (
    extended_spatial_analysis_df[
        "s_region"
    ]
    .apply(extract_region_geometry_type)
)

# RA and Dec together constitute one coordinate.
extended_spatial_analysis_df[
    "coordinate_key"
] = (
    extended_spatial_analysis_df[
        "s_ra"
    ]
    .round(10)
    .astype("string")
    +
    "|"
    +
    extended_spatial_analysis_df[
        "s_dec"
    ]
    .round(10)
    .astype("string")
)

In [83]:
spatial_required_columns = [
    "member_ous_uid",
    "obs_id_source",
    "coordinate_key",
    "s_region",
    "is_mosaic",
    "region_geometry_type",
]

missing_spatial_columns = [
    column
    for column in spatial_required_columns
    if column not in extended_spatial_analysis_df.columns
]

print(
    "Missing spatial columns:",
    missing_spatial_columns,
)

assert not missing_spatial_columns

Missing spatial columns: []


In [84]:
spatial_structure = (
    extended_spatial_analysis_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )
    .agg(
        source_count=(
            "obs_id_source",
            "nunique",
        ),
        coordinate_count=(
            "coordinate_key",
            "nunique",
        ),
        region_count=(
            "s_region",
            "nunique",
        ),
        mosaic_state_count=(
            "is_mosaic",
            "nunique",
        ),
        mosaic_state=(
            "is_mosaic",
            "first",
        ),
        geometry_count=(
            "region_geometry_type",
            "nunique",
        ),
    )
    .reset_index()
)

display(
    spatial_structure
    .sort_values(
        [
            "source_count",
            "region_count",
        ],
        ascending=False,
    )
    .head(20)
)

,member_ous_uid,source_count,coordinate_count,region_count,mosaic_state_count,mosaic_state,geometry_count
35,uid://A001/X3819/X186,36,22,36,1,F,1
27,uid://A001/X3788/Xc584,16,16,16,1,F,1
13,uid://A001/X3641/X98,10,10,10,1,F,1
65,uid://A001/X3833/X4d0,8,8,8,1,T,1
9,uid://A001/X15bf/Xc3,6,6,6,1,F,1
68,uid://A001/X3833/X5802,4,4,4,1,F,1
22,uid://A001/X3788/X7cc8,3,3,3,1,F,1
23,uid://A001/X3788/X7cca,3,3,3,1,F,1
26,uid://A001/X3788/Xb60e,3,3,3,1,F,1
67,uid://A001/X3833/X57f6,3,3,3,1,F,1


In [85]:
member_geometry_types = (
    extended_spatial_analysis_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )["region_geometry_type"]
    .apply(
        lambda values: tuple(
            sorted(
                values
                .dropna()
                .astype(str)
                .unique()
            )
        )
    )
    .rename("geometry_types")
    .reset_index()
)

spatial_structure = spatial_structure.merge(
    member_geometry_types,
    on="member_ous_uid",
    how="left",
    validate="one_to_one",
)

display(
    spatial_structure
    .sort_values(
        [
            "source_count",
            "region_count",
        ],
        ascending=False,
    )
    .head(20)
)

,member_ous_uid,source_count,coordinate_count,region_count,mosaic_state_count,mosaic_state,geometry_count,geometry_types
35,uid://A001/X3819/X186,36,22,36,1,F,1,"(Polygon,)"
27,uid://A001/X3788/Xc584,16,16,16,1,F,1,"(Polygon,)"
13,uid://A001/X3641/X98,10,10,10,1,F,1,"(Polygon,)"
65,uid://A001/X3833/X4d0,8,8,8,1,T,1,"(Polygon,)"
9,uid://A001/X15bf/Xc3,6,6,6,1,F,1,"(Circle,)"
68,uid://A001/X3833/X5802,4,4,4,1,F,1,"(Circle,)"
22,uid://A001/X3788/X7cc8,3,3,3,1,F,1,"(Polygon,)"
23,uid://A001/X3788/X7cca,3,3,3,1,F,1,"(Polygon,)"
26,uid://A001/X3788/Xb60e,3,3,3,1,F,1,"(Circle,)"
67,uid://A001/X3833/X57f6,3,3,3,1,F,1,"(Circle,)"


In [86]:
display(
    pd.crosstab(
        extended_spatial_analysis_df[
            "is_mosaic"
        ],
        extended_spatial_analysis_df[
            "region_geometry_type"
        ],
        margins=True,
    )
)

region_geometry_type,Circle,Polygon,Union,All
is_mosaic,,,,
F,236,618,0,854
T,0,358,14,372
All,236,976,14,1226


In [87]:
role_test_member_uids = (
    extended_member_summary
    .sort_values(
        [
            "asdm_count",
            "source_count",
            "archive_row_count",
        ],
        ascending=False,
    )
    ["member_ous_uid"]
    .dropna()
    .astype(str)
    .head(5)
    .tolist()
)

print("Role-test Member OUS:")
for member_uid in role_test_member_uids:
    print(member_uid)

Role-test Member OUS:
uid://A001/X3833/X1022
uid://A001/X3819/X186
uid://A001/X3788/Xc584
uid://A001/X3641/X98
uid://A001/X3833/X4d0


In [88]:
role_test_uid_sql = ",\n        ".join(
    f"'{member_uid}'"
    for member_uid in role_test_member_uids
)

In [89]:
role_test_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE member_ous_uid IN (
        {role_test_uid_sql}
)
"""

role_test_count_df = run_tap_query(
    role_test_count_query,
    maxrec=1,
)

role_test_expected_rows = int(
    role_test_count_df[
        "total_rows"
    ].iloc[0]
)

print(
    "Expected role-test rows:",
    role_test_expected_rows,
)

Expected role-test rows: 836


In [90]:
role_test_query = f"""
SELECT
    proposal_id,
    member_ous_uid,
    asdm_uid,
    obs_id,
    target_name,
    science_observation,
    type,
    scan_intent,
    dataproduct_type,
    calib_level,
    data_rights,
    obs_release_date,
    qa2_passed
FROM ivoa.obscore
WHERE member_ous_uid IN (
        {role_test_uid_sql}
)
"""

role_test_df = run_tap_query(
    role_test_query,
    maxrec=max(
        role_test_expected_rows,
        1,
    ),
)

print(
    "Retrieved role-test rows:",
    len(role_test_df),
)
print(
    "Complete role-test retrieval:",
    len(role_test_df)
    == role_test_expected_rows,
)

Retrieved role-test rows: 836
Complete role-test retrieval: True


In [91]:
science_state_summary = (
    role_test_df
    .groupby(
        "science_observation",
        dropna=False,
    )
    .agg(
        archive_rows=("obs_id", "size"),
        member_count=(
            "member_ous_uid",
            "nunique",
        ),
        target_count=(
            "target_name",
            "nunique",
        ),
    )
    .reset_index()
)

display(science_state_summary)

,science_observation,archive_rows,member_count,target_count
0,F,236,4,21
1,T,600,5,72


In [92]:
observation_role_summary = (
    role_test_df
    .groupby(
        [
            "science_observation",
            "type",
            "scan_intent",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="archive_row_count"
    )
    .sort_values(
        "archive_row_count",
        ascending=False,
    )
)

display(
    observation_role_summary.head(30)
)

,science_observation,type,scan_intent,archive_row_count
8,T,S,TARGET,600
1,F,S,BANDPASS FLUX WVR,82
0,F,S,BANDPASS DIFFGAIN FLUX WVR,52
6,F,S,PHASE WVR,38
3,F,S,CHECK WVR,32
4,F,S,DIFFGAIN WVR,12
5,F,S,FLUX WVR,8
7,F,S,POLARIZATION WVR,8
2,F,S,BANDPASS WVR,4


In [93]:
role_test_df[
    "release_date_is_3000_sentinel"
] = (
    role_test_df[
        "obs_release_date"
    ]
    .astype("string")
    .str.startswith(
        "3000-01-01",
        na=False,
    )
)

release_state_summary = (
    role_test_df
    .groupby(
        [
            "data_rights",
            "qa2_passed",
            "release_date_is_3000_sentinel",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="archive_row_count"
    )
    .sort_values(
        "archive_row_count",
        ascending=False,
    )
)

display(release_state_summary)

,data_rights,qa2_passed,release_date_is_3000_sentinel,archive_row_count
0,Proprietary,F,True,836


In [94]:
publisher_matches_proposal = (
    extended_recent_df["obs_publisher_did"]
    ==
    (
        "ADS/JAO.ALMA#"
        + extended_recent_df["proposal_id"].astype(str)
    )
)

print(
    "Publisher DID matches proposal:",
    publisher_matches_proposal.sum(),
    "/",
    len(publisher_matches_proposal),
)

Publisher DID matches proposal: 1226 / 1226


# Experimental Analysis and Conclusions

## 1. Scope

This notebook investigated ALMA Archive row granularity, identifier scope, and
relationships between Proposal, Group OUS, Member OUS, ASDM, source, and
spectral-window metadata.

| Structural sample | Member OUS datasets | Archive rows |
| --- | ---: | ---: |
| Original stratified sample | 24 | 193 |
| Extended recent sample | 96 | 1,226 |
| Total | 120 | 1,419 |

All expected rows were retrieved without truncation. All 1,419 tested `obs_id`
values were successfully parsed into source and spectral-window components.
Additional proposal-level, schema, spatial, and science/calibration queries were
used to test specific relationships but were not added to this total.

## 2. Archive-Row Granularity

All 120 tested Member OUS datasets satisfied:

$$
N_{\mathrm{rows}}
=
N_{\mathrm{source\ contexts}}
\times
N_{\mathrm{SPWs}}
$$

This included complex examples such as 16 sources × 16 SPWs, 36 sources ×
4 SPWs, and 8 sources × 14 SPWs.

In the 1,226-row extended sample, both `obs_id` and
`(member_ous_uid, obs_id_source, spw_identifier)` were unique. The tested
records therefore formed complete source–SPW grids.

The current sample-supported interpretation is:

> An Archive row represents a source–spectral-window-related record within a
> Member OUS, rather than one complete observation.

The Notebook 1 relationship $N_{\mathrm{rows}}=N_{\mathrm{SPWs}}$ is the
single-source special case. These identifiers remain candidate keys rather than
confirmed Archive primary keys.

## 3. Identifier Scope and Hierarchy

The tested data supported the directional hierarchy:

`Proposal → Group OUS → Member OUS → ASDM`

No violations were found for Member OUS → Proposal, Member OUS → Group OUS,
Group OUS → Proposal, or ASDM → Member OUS. The relationships are not
one-to-one: proposals and groups may contain multiple lower-level datasets, and
one Member OUS may contain multiple ASDMs. Empty `group_ous_uid` values should
be normalized to missing values.

In the extended sample, `proposal_id` and `obs_publisher_did` both had 59 unique
values, while there were 96 Member OUS datasets and 1,226 `obs_id` values.
`obs_publisher_did` behaved as a proposal-scoped identifier and could span
multiple Members and rows. It must not be used as an Archive-row or Member-level
primary key.

## 4. Member OUS, Source Context, and Spatial Metadata

Member OUS is a useful dataset container, but many observational fields are not
single-valued within it. A provisional **Source Context** is therefore defined
as:

`(member_ous_uid, parsed obs_id source)`

This is an internal grouping concept, not an official ALMA Archive entity.

The experiments also showed that source identity, coordinates, footprint, and
mosaic state are different concepts:

- different source labels may refer to the same physical target;
- the same physical target may occur in different observational contexts;
- multiple source contexts may share one coordinate;
- one Member contained 36 source contexts, 22 coordinates, and 36 footprints;
- non-mosaic records may have Polygon footprints;
- mosaic and multi-source status are independent;
- several Members contained more than one `is_mosaic` or geometry value.

Therefore, `target_name`, coordinates, `s_region`, and `is_mosaic` must not be
used interchangeably. Mosaic state and footprint should remain attached to the
relevant source or Archive record; a Member-level state may be derived as
non-mosaic, mosaic, mixed, or unknown.

## 5. Spectral Windows, ASDM, and Frequency

TAP_SCHEMA confirmed the relevant units:

| Field | Unit | Interpretation |
| --- | --- | --- |
| `frequency` | GHz | observed sky reference frequency |
| `bandwidth` | Hz | total bandwidth |
| `frequency_support` | structured text | complete frequency coverage and associated metadata |

Spectral windows must be represented as variable-length collections. In the
initial validation samples, SPW counts matched the number of parsed
`frequency_support` intervals.

Exact frequency was not determined by `(Member, SPW)` or `(ASDM, SPW)`.
Source-dependent frequency spreads ranged from approximately 0.027 MHz to
5.52 MHz, while bandwidth remained stable in the tested cases. The model should
therefore separate a logical SPW and nominal bandwidth from source-specific
exact frequency.

One Member contained two sources, four SPWs, two ASDMs, and eight rows. The two
ASDMs shared the same nominal SPW structure and bandwidth but differed in source,
antenna list, exact frequency, sensitivity, time metadata, and complete
`frequency_support` string.

Because `frequency_support` also includes resolution, sensitivity, and
polarization, exact string equality is too strict for frequency-coverage
comparison. Its components must be parsed and compared separately.

## 6. Observation Role and Data Quality

A targeted unfiltered query returned 836 rows:

| `science_observation` | Rows | Interpretation |
| --- | ---: | --- |
| `T` | 600 | science targets with `scan_intent=TARGET` |
| `F` | 236 | calibration and checking intents |

The non-science rows included BANDPASS, FLUX, PHASE, CHECK, DIFFGAIN,
POLARIZATION, and WVR intents. Candidate retrieval for duplication assessment
should therefore retain the `science_observation='T'` filter.

Twenty-four original rows used an empty string rather than NULL for
`group_ous_uid`. In the role-test sample, all 836 rows simultaneously had
`data_rights=Proprietary`, `qa2_passed=F`, and an `obs_release_date` beginning
with `3000-01-01`. This date should be treated as an unconfirmed sentinel rather
than an ordinary chronological value. The raw release date, access rights, and
QA status must be stored separately.

## 7. Data-Model Implications and Next Step

The evidence supports the preliminary structure:

`Proposal → optional Group OUS → Member OUS → Source Context × Logical SPW → Archive Row`

The internal model should distinguish:

1. **Raw Archive Row** — preserve original metadata and identifiers.
2. **Member OUS** — group related Archive records.
3. **ASDM association** — allow one Member to reference multiple ASDMs.
4. **Source Context** — retain source label, coordinates, footprint, mosaic
   state, sensitivity, and provenance.
5. **Logical Spectral Window** — store SPW identifier and nominal bandwidth.
6. **Source–SPW Record** — store exact frequency and other row-level values.
7. **Physical-Target Candidate** — support later normalization without
   overwriting raw source labels.

The samples were selected to expose structural cases and are not statistically
representative of the complete Archive. All conclusions remain limited to the
tested records.

Notebook 2 establishes a sufficiently strong preliminary relationship model.
The next notebook should investigate `frequency_support` parsing, unit
conversion, frequency resolution, sensitivity, polarization, and
tolerance-aware frequency comparison.

## Additional Validation of Source Context

The provisional Source Context is defined as:

`(member_ous_uid, obs_id_source)`

The following experiment tests which Archive fields are functionally stable
within this grouping.

In [96]:
source_context_required_columns = [
    "member_ous_uid",
    "obs_id_source",
    "target_name",
    "coordinate_key",
    "s_region",
    "is_mosaic",
    "region_geometry_type",
    "asdm_uid",
    "antenna_arrays",
    "frequency_range_signature",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
]

missing_source_context_columns = [
    column
    for column in source_context_required_columns
    if column not in extended_spatial_analysis_df.columns
]

print(
    "Missing Source Context columns:",
    missing_source_context_columns,
)

assert not missing_source_context_columns

Missing Source Context columns: []


In [97]:
source_context_dependency_checks = pd.DataFrame(
    [
        {
            "relationship":
                "(Member, source) -> target_name",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "target_name",
                ),
        },
        {
            "relationship":
                "(Member, source) -> coordinate",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "coordinate_key",
                ),
        },
        {
            "relationship":
                "(Member, source) -> s_region",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "s_region",
                ),
        },
        {
            "relationship":
                "(Member, source) -> is_mosaic",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "is_mosaic",
                ),
        },
        {
            "relationship":
                "(Member, source) -> geometry type",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "region_geometry_type",
                ),
        },
        {
            "relationship":
                "(Member, source) -> ASDM",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "asdm_uid",
                ),
        },
        {
            "relationship":
                "(Member, source) -> antenna arrays",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "antenna_arrays",
                ),
        },
        {
            "relationship":
                "(Member, source) -> frequency ranges",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "frequency_range_signature",
                ),
        },
        {
            "relationship":
                "(Member, source) -> spatial resolution",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "spatial_resolution",
                ),
        },
        {
            "relationship":
                "(Member, source) -> line sensitivity",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "sensitivity_10kms",
                ),
        },
        {
            "relationship":
                "(Member, source) -> continuum sensitivity",
            "violations":
                count_dependency_violations(
                    extended_spatial_analysis_df,
                    [
                        "member_ous_uid",
                        "obs_id_source",
                    ],
                    "cont_sensitivity_bandwidth",
                ),
        },
    ]
)

display(source_context_dependency_checks)

,relationship,violations
0,"(Member, source) -> target_name",0
1,"(Member, source) -> coordinate",0
2,"(Member, source) -> s_region",0
3,"(Member, source) -> is_mosaic",0
4,"(Member, source) -> geometry type",0
5,"(Member, source) -> ASDM",0
6,"(Member, source) -> antenna arrays",0
7,"(Member, source) -> frequency ranges",0
8,"(Member, source) -> spatial resolution",0
9,"(Member, source) -> line sensitivity",184


In [99]:
mixed_mosaic_member_uids = (
    spatial_structure.loc[
        spatial_structure[
            "mosaic_state_count"
        ] > 1,
        "member_ous_uid",
    ]
    .dropna()
    .tolist()
)

print(
    "Members with mixed mosaic state:",
    len(mixed_mosaic_member_uids),
)

Members with mixed mosaic state: 4


In [100]:
for member_uid in mixed_mosaic_member_uids:
    member_rows = (
        extended_spatial_analysis_df.loc[
            extended_spatial_analysis_df[
                "member_ous_uid"
            ] == member_uid,
            [
                "member_ous_uid",
                "obs_id_source",
                "spw_identifier",
                "target_name",
                "is_mosaic",
                "region_geometry_type",
                "coordinate_key",
                "s_region",
            ],
        ]
        .sort_values(
            [
                "obs_id_source",
                "spw_identifier",
            ]
        )
    )

    print("\nMember OUS:", member_uid)

    display(
        pd.crosstab(
            member_rows["obs_id_source"],
            member_rows["is_mosaic"],
            margins=True,
        )
    )

    display(
        pd.crosstab(
            member_rows["obs_id_source"],
            member_rows[
                "region_geometry_type"
            ],
            margins=True,
        )
    )


Member OUS: uid://A001/X3833/X2768


is_mosaic,F,T,All
obs_id_source,,,
1E1740.7-2942,10,0,10
1E1740.7-2942_OFF_0,0,10,10
All,10,10,20


region_geometry_type,Polygon,Union,All
obs_id_source,,,
1E1740.7-2942,10,0,10
1E1740.7-2942_OFF_0,0,10,10
All,10,10,20



Member OUS: uid://A001/X383d/X43a


is_mosaic,F,T,All
obs_id_source,,,
NGC_0300,0,4,4
NGC_0300_FluxRef,4,0,4
All,4,4,8


...

is_mosaic,F,T,All
obs_id_source,,,
NGC_0300,0,4,4
NGC_0300_FluxRef,4,0,4
All,4,4,8


region_geometry_type,Circle,Polygon,All
obs_id_source,,,
NGC_0300,0,4,4
NGC_0300_FluxRef,4,0,4
All,4,4,8



Member OUS: uid://A001/X383d/X462


is_mosaic,F,T,All
obs_id_source,,,
NGC_0300,0,4,4
NGC_0300_FluxRef,4,0,4
All,4,4,8


region_geometry_type,Circle,Polygon,All
obs_id_source,,,
NGC_0300,0,4,4
NGC_0300_FluxRef,4,0,4
All,4,4,8


In [101]:
mixed_mosaic_context_summary = (
    extended_spatial_analysis_df.loc[
        extended_spatial_analysis_df[
            "member_ous_uid"
        ].isin(mixed_mosaic_member_uids)
    ]
    .groupby(
        [
            "member_ous_uid",
            "obs_id_source",
        ],
        dropna=False,
    )
    .agg(
        row_count=("obs_id", "size"),
        spw_count=(
            "spw_identifier",
            "nunique",
        ),
        mosaic_state_count=(
            "is_mosaic",
            "nunique",
        ),
        geometry_count=(
            "region_geometry_type",
            "nunique",
        ),
        region_count=(
            "s_region",
            "nunique",
        ),
    )
    .reset_index()
)

display(mixed_mosaic_context_summary)

,member_ous_uid,obs_id_source,row_count,spw_count,mosaic_state_count,geometry_count,region_count
0,uid://A001/X3833/X2768,1E1740.7-2942,10,10,1,1,1
1,uid://A001/X3833/X2768,1E1740.7-2942_OFF_0,10,10,1,1,1
2,uid://A001/X383d/X43a,NGC_0300,4,4,1,1,1
3,uid://A001/X383d/X43a,NGC_0300_FluxRef,4,4,1,1,1
4,uid://A001/X383d/X45a,NGC_0300,4,4,1,1,1
5,uid://A001/X383d/X45a,NGC_0300_FluxRef,4,4,1,1,1
6,uid://A001/X383d/X462,NGC_0300,4,4,1,1,1
7,uid://A001/X383d/X462,NGC_0300_FluxRef,4,4,1,1,1


In [102]:
role_test_parsed_df = (
    role_test_df["obs_id"]
    .astype("string")
    .str.extract(obs_id_pattern)
)

role_test_analysis_df = pd.concat(
    [
        role_test_df.reset_index(drop=True),
        role_test_parsed_df.reset_index(
            drop=True
        ),
    ],
    axis=1,
)

role_candidate_keys = pd.DataFrame(
    [
        evaluate_candidate_key(
            role_test_analysis_df,
            "obs_id",
            ["obs_id"],
        ),
        evaluate_candidate_key(
            role_test_analysis_df,
            "Member + source + SPW",
            [
                "member_ous_uid",
                "obs_id_source",
                "spw_identifier",
            ],
        ),
        evaluate_candidate_key(
            role_test_analysis_df,
            "Member + source + SPW + role",
            [
                "member_ous_uid",
                "obs_id_source",
                "spw_identifier",
                "science_observation",
            ],
        ),
    ]
)

display(role_candidate_keys)

,candidate_key,columns,total_rows,unique_key_count,duplicate_key_groups,rows_in_duplicate_groups,is_unique_in_sample
0,obs_id,obs_id,836,836,0,0,True
1,Member + source + SPW,"member_ous_uid, obs_id_source, spw_identifier",836,836,0,0,True
2,Member + source + SPW + role,"member_ous_uid, obs_id_source, spw_identifier, science_observation",836,836,0,0,True


A functional-dependency audit on all 1,226 extended-sample rows found no
violations for Source Context determining target name, coordinates, spatial
footprint, mosaic state, geometry type, ASDM, antenna arrays, frequency-range
signature, spatial resolution, or continuum sensitivity.

However, `(Member, source) -> sensitivity_10kms` produced 184 violations.
Continuum sensitivity may therefore be represented at Source Context level in
the tested model, while line sensitivity must remain associated with the
source–SPW Archive record.

Four Member OUS datasets contained mixed mosaic states at Member level. In all
eight affected Source Contexts, mosaic state, geometry type, and spatial region
were individually stable. The mixed Member-level state was caused by different
source contexts rather than variation between SPWs of the same source.

Candidate-key tests on the 836 unfiltered science and calibration rows found no
duplicate `obs_id`, Member–source–SPW, or Member–source–SPW–role combinations.
Observation role was therefore not required to make the tested row keys unique.